# Adaptive Signals — Stage 2 · Live simulation

Run the **first two code cells**, then press **Start / reset run** in the viewer. No GPU, input ZIP, GitHub checkout, API key or public tunnel is needed. Keep the Python/Colab runtime connected.

The interface draws actual SUMO vehicle positions and signal states. Pause, step one second, reset the traffic seed, compare controllers, adjust demand, or corrupt the controller's detector counts. This is a local runtime application, not a permanently hosted website.

The accepted experiment contains **284 evaluation runs and 48 development runs** on a synthetic nine-intersection grid. The short-horizon controller is a C++ heuristic; there is no RL training, live CCTV or calibrated Thai traffic in this stage.

The optional third cell shows the saved summary and can rerun/export the benchmark. It is disabled by default so the live viewer opens promptly. Colab uses its private iframe helper. Backend behavior is tested; browser visual QA and the actual Colab iframe/proxy are not claimed tested.


In [ ]:
# Cell 1 — self-contained source, pinned dependencies, C++ build, checks.
import os, sys, subprocess, shutil, time, json, urllib.request, zipfile
from pathlib import Path
if "LIVE_PROCESS" in globals() and LIVE_PROCESS.poll() is None:
    LIVE_PROCESS.terminate()
    LIVE_PROCESS.wait(timeout=10)
BASE=Path(os.environ.get("ADAPTIVE_SIGNALS_HOME", "/content" if Path("/content").exists() else str(Path.cwd())))
PROJECT=BASE/"AdaptiveSignals_Stage2"
PROJECT.mkdir(parents=True,exist_ok=True)
PAYLOAD={'experiment.py': '"""Reproducible SUMO pilot: synthetic left-hand grid, C++ control kernels.\n\nNo RL training, real CCTV, calibrated Thai demand, or deployment claims.\nRun `python experiment.py --help` for smoke, pilot and extended campaigns.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport bisect\nimport csv\nimport ctypes as ct\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport random\nimport statistics\nimport subprocess\nimport sys\nimport time\nimport xml.etree.ElementTree as ET\nfrom collections import defaultdict, deque\n\nimport numpy as np\nimport sumolib\nimport libsumo as sim\n\nROOT = Path(__file__).resolve().parent\nCONTROLLERS = [\'fixed\', \'queue_actuated\', \'max_pressure\', \'receiving_aware\', \'coordinated\']\nMODE = {\'queue_actuated\':0,\'max_pressure\':1,\'receiving_aware\':2,\'coordinated\':3,\n        \'calibrated_pressure\':1,\'message_only\':3,\'moving_only\':3,\'predictive\':3}\nSCENARIOS = [\'balanced\', \'directional_switch\', \'surge\']\nSETTINGS = {\n    \'version\': \'stage2-v1\', \'grid\': 3, \'spacing_m\': 160, \'boundary_m\': 220,\n    \'phase_layout\': \'incoming\',\n    \'step_s\': 1, \'minimum_green_s\': 10, \'maximum_green_s\': 50,\n    \'yellow_s\': 3, \'all_red_s\': 2, \'decision_interval_s\': 5, \'maximum_red_target_s\':180,\n    \'hysteresis_vehicles\': 2.0, \'vehicle_length_m\': 5, \'min_gap_m\': 2.5,\n    \'observation_sample_s\': 5, \'observation_delay_s\': 5,\n    \'dropout_probability\': 0.10, \'outage_fraction\': [0.40, 0.55],\n    \'coordination_weight\': 0.30, \'moving_weight\': 0.20,\n    \'fixed_candidates_s\': [20, 30, 45], \'development_seed\': 101,\n    \'evaluation_seeds\': [4777, 5777, 6777],\n}\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\')\n    temp.replace(path)\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef binary(name):\n    import sumo\n    candidate = Path(sumo.__file__).parent / \'bin\' / name\n    return str(candidate) if candidate.exists() else sumolib.checkBinary(name)\n\n\nclass Kernel:\n    def __init__(self, config=None):\n        self.config=config or {\'horizon_s\':20,\'receiving_power\':1,\'message_weight\':0.3,\'moving_weight\':0.2}\n        source = ROOT / \'src/controllers.cpp\'\n        library = ROOT / \'build/controllers.so\'\n        library.parent.mkdir(exist_ok=True)\n        if not library.exists() or library.stat().st_mtime < source.stat().st_mtime:\n            subprocess.run([\'g++\', \'-std=c++17\', \'-O3\', \'-Wall\', \'-Wextra\',\n                            \'-Werror\', \'-shared\', \'-fPIC\', str(source), \'-o\', str(library)], check=True)\n        self.lib = ct.CDLL(str(library))\n        ptr = ct.POINTER(ct.c_double)\n        self.lib.phase_score.argtypes = [ct.c_int, ct.c_int] + [ptr]*5\n        self.lib.phase_score.restype = ct.c_double\n        self.lib.should_switch.argtypes = [ct.c_double]*6\n        self.lib.should_switch.restype = ct.c_int\n        self.lib.discharge_score.argtypes=[ct.c_int]+[ptr]*5+[ct.c_double]*5\n        self.lib.discharge_score.restype=ct.c_double\n\n    def score(self, mode, rows):\n        if not rows:\n            return 0.0\n        arrays = [np.ascontiguousarray(a, dtype=np.float64) for a in zip(*rows)]\n        return self.lib.phase_score(mode, len(rows), *[\n            a.ctypes.data_as(ct.POINTER(ct.c_double)) for a in arrays])\n\n    def switch(self, current, next_, elapsed):\n        return bool(self.lib.should_switch(current, next_, elapsed,\n            SETTINGS[\'minimum_green_s\'], SETTINGS[\'maximum_green_s\'],\n            SETTINGS[\'hysteresis_vehicles\']))\n\n    def predictive(self,rows,lost_time):\n        if not rows:return 0.0\n        arrays=[np.ascontiguousarray(a,dtype=np.float64) for a in zip(*rows)]\n        c=self.config\n        return self.lib.discharge_score(len(rows),*[a.ctypes.data_as(ct.POINTER(ct.c_double)) for a in arrays],\n            c[\'horizon_s\'],lost_time,c[\'receiving_power\'],c[\'message_weight\'],c[\'moving_weight\'])\n\n\ndef build_network(folder):\n    folder.mkdir(parents=True, exist_ok=True)\n    netfile = folder / \'grid.net.xml\'\n    if not netfile.exists():\n        command = [binary(\'netgenerate\'), \'--grid\', \'--grid.number\', str(SETTINGS[\'grid\']),\n            \'--grid.length\', str(SETTINGS[\'spacing_m\']), \'--grid.attach-length\', str(SETTINGS[\'boundary_m\']),\n            \'--default-junction-type\', \'traffic_light\', \'--default.lanenumber\', \'1\',\n            \'--default.speed\', \'13.89\', \'--no-turnarounds\', \'--lefthand\',\n            \'--tls.layout\', SETTINGS[\'phase_layout\'],\n            \'--tls.yellow.time\', str(SETTINGS[\'yellow_s\']),\n            \'--tls.allred.time\', str(SETTINGS[\'all_red_s\']),\n            \'--tls.green.time\', \'30\', \'--output-file\', str(netfile)]\n        p = subprocess.run(command, text=True, capture_output=True, check=True)\n        (folder / \'netgenerate.log\').write_text(p.stdout + p.stderr)\n    net = sumolib.net.readNet(str(netfile))\n    xml = ET.parse(netfile).getroot()\n    tls = {}\n    for logic in xml.findall(\'tlLogic\'):\n        phases = [{\'state\': p.attrib[\'state\'], \'duration\': float(p.attrib[\'duration\'])}\n                  for p in logic.findall(\'phase\')]\n        greens = [i for i,p in enumerate(phases) if any(c in \'Gg\' for c in p[\'state\'])]\n        if len(greens) != 4:\n            raise ValueError(\'Stage 1 requires four protected incoming-approach stages\')\n        assert not any(\'g\' in p[\'state\'] for p in phases), \'Unexpected permissive green\'\n        for i in greens:\n            yellow = phases[(i+1)%len(phases)]\n            red = phases[(i+2)%len(phases)]\n            assert \'y\' in yellow[\'state\'] and yellow[\'duration\'] == SETTINGS[\'yellow_s\']\n            assert set(red[\'state\']) == {\'r\'} and red[\'duration\'] == SETTINGS[\'all_red_s\']\n        tls[logic.attrib[\'id\']] = {\'phases\': phases, \'greens\': greens}\n    assert len(tls) == SETTINGS[\'grid\']**2\n    return netfile, net, tls\n\n\ndef generate_demand(folder, net, scenario, seed, demand_s, scale=1.0):\n    """Explicit immutable vehicle requests; routes and departures shared by controllers."""\n    rng = random.Random(seed)\n    entries = [e for e in net.getEdges() if len(e.getFromNode().getIncoming()) == 1]\n    entries.sort(key=lambda e:e.getID())\n    routes = []\n    root = ET.Element(\'routes\')\n    ET.SubElement(root, \'vType\', id=\'car\', length=\'5\', minGap=\'2.5\',\n                  accel=\'2.6\', decel=\'4.5\', sigma=\'0\', speedFactor=\'1.0\', maxSpeed=\'13.89\')\n    requests = []\n    for origin in entries:\n        x,y = origin.getFromNode().getCoord()\n        # Opposite boundary with turns through different streets where feasible.\n        if origin.getID().startswith(\'left\'): choices = [e for e in entries if e.getID().startswith(\'right\')]\n        elif origin.getID().startswith(\'right\'): choices = [e for e in entries if e.getID().startswith(\'left\')]\n        elif origin.getID().startswith(\'bottom\'): choices = [e for e in entries if e.getID().startswith(\'top\')]\n        else: choices = [e for e in entries if e.getID().startswith(\'bottom\')]\n        for dest_entry in choices:\n            exits = dest_entry.getFromNode().getIncoming()\n            target = exits[0]\n            path, _ = net.getShortestPath(origin, target)\n            if not path:\n                raise RuntimeError(\'Unreachable OD pair\')\n            routes.append({\'origin\': origin.getID(), \'target\': target.getID(),\n                           \'edges\': \' \'.join(e.getID() for e in path),\n                           \'axis\': \'horizontal\' if origin.getID().startswith((\'left\',\'right\')) else \'vertical\'})\n        group = routes[-len(choices):]\n        for second in range(demand_s):\n            frac = second / demand_s\n            axis = group[0][\'axis\']\n            if scenario == \'balanced\': rate = 0.105\n            elif scenario == \'directional_switch\':\n                dominant = \'horizontal\' if frac < 0.5 else \'vertical\'\n                rate = 0.19 if axis == dominant else 0.055\n            elif scenario == \'surge\':\n                rate = 0.25 if 0.30 <= frac < 0.65 else 0.085\n            else: raise ValueError(scenario)\n            if rng.random() < min(1.0,rate*scale):\n                route = rng.choice(group)\n                requests.append((second + rng.random(), route[\'edges\']))\n    requests.sort()\n    for i, (depart, edges) in enumerate(requests):\n        # Assign driver speeds explicitly before simulation: controllers cannot\n        # change which random speed factor a later-inserted vehicle receives.\n        veh = ET.SubElement(root, \'vehicle\', id=f\'v{i:06d}\', type=\'car\',\n            depart=f\'{depart:.3f}\', departLane=\'best\', departSpeed=\'max\',\n            speedFactor=f\'{rng.uniform(0.90,1.10):.5f}\')\n        ET.SubElement(veh, \'route\', edges=edges)\n    demand = folder / f\'{scenario}_seed{seed}_d{demand_s}_load{scale:g}.rou.xml\'\n    ET.indent(root)\n    ET.ElementTree(root).write(demand, encoding=\'utf-8\', xml_declaration=True)\n    schedule = {v.attrib[\'id\']:float(v.attrib[\'depart\']) for v in root.findall(\'vehicle\')}\n    return demand, schedule\n\n\ndef get_layout(tls):\n    lanes = set()\n    for tid, item in tls.items():\n        links = sim.trafficlight.getControlledLinks(tid)\n        maps = {}\n        for phase in item[\'greens\']:\n            served = defaultdict(set)\n            for color, group in zip(item[\'phases\'][phase][\'state\'], links):\n                if color in \'Gg\':\n                    for incoming, outgoing, via in group:\n                        served[incoming].add(outgoing)\n                        lanes.update([incoming, outgoing])\n            maps[phase] = {k:sorted(v) for k,v in served.items()}\n        item[\'maps\'] = maps\n    capacities = {lane:sim.lane.getLength(lane)/7.5 for lane in sorted(lanes)}\n    return sorted(lanes), capacities\n\n\nclass Observation:\n    """Counts are sample-and-hold, delayed, noisy and missing in degraded mode.\n\n    Detector noise uses a separate seeded RNG, identical draw order across\n    controllers. Evaluation metrics always use simulator truth.\n    """\n    def __init__(self, lanes, capacities, regime, seed, demand_s):\n        self.lanes, self.capacities, self.regime = lanes, capacities, regime\n        self.rng = random.Random(seed+950001)\n        self.history = deque()\n        self.current = {l:(0.0,0.0) for l in lanes} # queue, vehicle count\n        self.measured = self.current.copy()\n        self.demand_s = demand_s\n        self.outage_lanes = set(l for l in lanes if l.startswith(\'B1\'))\n        self.stats = {\'sample_lane_opportunities\':0, \'dropped_lane_samples\':0}\n\n    def update(self,t):\n        if self.regime == \'perfect\':\n            self.current = {l:(float(sim.lane.getLastStepHaltingNumber(l)),\n                               float(sim.lane.getLastStepVehicleNumber(l))) for l in self.lanes}\n            return self.current\n        if int(t) % SETTINGS[\'observation_sample_s\'] == 0:\n            for lane in self.lanes:\n                self.stats[\'sample_lane_opportunities\'] += 1\n                drop = self.rng.random() < SETTINGS[\'dropout_probability\']\n                zq, zn = self.rng.gauss(0,1), self.rng.gauss(0,1)\n                outage = (self.demand_s*0.40 <= t < self.demand_s*0.55 and lane in self.outage_lanes)\n                if drop or outage:\n                    self.stats[\'dropped_lane_samples\'] += 1\n                    continue\n                q = sim.lane.getLastStepHaltingNumber(lane)\n                n = sim.lane.getLastStepVehicleNumber(lane)\n                n = max(0.0,min(self.capacities[lane], n + (0.20*n+0.5)*zn))\n                q = max(0.0,min(n, q + (0.20*q+0.5)*zq))\n                self.measured[lane] = (q,n)\n            self.history.append((t+SETTINGS[\'observation_delay_s\'], self.measured.copy()))\n        while self.history and self.history[0][0] <= t:\n            _, self.current = self.history.popleft()\n        return self.current\n\n\ndef announcements(tls, states, obs):\n    """One-hop message: queued demand currently being served toward each lane."""\n    message = defaultdict(float)\n    for tid,item in tls.items():\n        phase = states[tid][\'phase\']\n        if phase not in item[\'maps\']:\n            continue\n        for incoming, outs in item[\'maps\'][phase].items():\n            for out in outs:\n                message[out] += obs[incoming][0] / len(outs)\n    return message\n\n\ndef phase_rows(mapping, obs, capacities, message, turn_weights=None):\n    rows = []\n    for incoming, outs in sorted(mapping.items()):\n        q,n = obs[incoming]\n        # Equal turning weights are an explicit approximation in this pilot.\n        weights=[(turn_weights or {}).get(incoming,{}).get(out,0.0) for out in outs]\n        if sum(weights)<=0:weights=[1/len(outs)]*len(outs)\n        else:weights=[w/sum(weights) for w in weights]\n        downstream_q = sum(w*obs[out][0] for w,out in zip(weights,outs))\n        downstream_fill = sum(w*min(1.0,obs[out][1]/capacities[out]) for w,out in zip(weights,outs))\n        rows.append((q, downstream_q, downstream_fill, message[incoming], max(0.0,n-q)))\n    return rows\n\n\ndef run_one(out, netfile, tls_template, demand, schedule, controller, regime,\n            seed, demand_s, drain_s, fixed_green, trace=True, config=None,\n            turn_weights=None, stop_when_clear=False, live_hook=None):\n    out.mkdir(parents=True, exist_ok=True)\n    # Copy structure because phase maps are attached by this run.\n    tls = json.loads(json.dumps(tls_template))\n    kernel = Kernel(config)\n    horizon = demand_s + drain_s\n    command = [binary(\'sumo\'), \'-n\', str(netfile), \'-r\', str(demand),\n        \'--seed\', str(seed), \'--step-length\', \'1\', \'--time-to-teleport\', \'-1\',\n        \'--max-depart-delay\', \'-1\', \'--collision.action\', \'warn\',\n        \'--collision.check-junctions\', \'true\', \'--no-step-log\', \'true\',\n        \'--duration-log.disable\', \'true\',\n        \'--tripinfo-output\', str(out/\'tripinfo.xml\'), \'--tripinfo-output.write-unfinished\', \'true\',\n        \'--error-log\', str(out/\'sumo.log\')]\n    started = time.monotonic()\n    sim.start(command)\n    departed, arrived = {}, {}\n    collisions = teleports = switches = 0\n    phases_trace, timeline, replay = [], [], []\n    try:\n        lanes, capacities = get_layout(tls)\n        observer = Observation(lanes, capacities, regime, seed, demand_s)\n        native=controller==\'sumo_actuated\'\n        states = {tid:{\'phase\':0, \'start\':0.0,\'target\':None,\n                      \'last_end\':{g:0.0 for g in item[\'greens\']}} for tid,item in tls.items()}\n        for tid in tls:\n            if native:\n                logic=sim.trafficlight.getAllProgramLogics(tid)[0]\n                # A new ID is required: updating an existing static program\'s\n                # type does not replace its underlying SUMO controller class.\n                logic.programID=\'reference_actuated\'\n                logic.type=3 # SUMO actuated, built-in induction-loop gap control\n                for phase in logic.phases:\n                    if \'G\' in phase.state:\n                        phase.minDur=SETTINGS[\'minimum_green_s\'];phase.maxDur=SETTINGS[\'maximum_green_s\']\n                sim.trafficlight.setProgramLogic(tid,logic)\n                sim.trafficlight.setProgram(tid,\'reference_actuated\')\n                sim.trafficlight.setPhase(tid,0)\n                sim.trafficlight.setPhaseDuration(tid,SETTINGS[\'minimum_green_s\'])\n                continue\n            sim.trafficlight.setPhase(tid,0)\n            sim.trafficlight.setPhaseDuration(tid,1e7)\n        for t in range(horizon):\n            if live_hook:live_hook(\'before\',t,states,lanes,obs if t else {},None)\n            obs = observer.update(t)\n            message = announcements(tls,states,obs)\n            updates = []\n            for tid,item in tls.items():\n                st = states[tid]\n                if native:\n                    actual=sim.trafficlight.getPhase(tid)\n                    if actual!=st[\'phase\']:\n                        old=st[\'phase\'];duration=t-st[\'start\']\n                        assert actual==(old+1)%len(item[\'phases\'])\n                        if old in item[\'greens\']:assert 10<=duration<=50\n                        else:assert duration==item[\'phases\'][old][\'duration\']\n                        phases_trace.append([t,tid,old,actual,duration])\n                        st.update(phase=actual,start=float(t))\n                        switches+=int(actual in item[\'greens\'])\n                    continue\n                phase, elapsed = st[\'phase\'], t-st[\'start\']\n                p = item[\'phases\'][phase]\n                change = False\n                target = None\n                if phase not in item[\'greens\']:\n                    change = elapsed >= p[\'duration\']\n                elif controller == \'fixed\':\n                    change = elapsed >= fixed_green\n                elif int(elapsed) % SETTINGS[\'decision_interval_s\'] == 0:\n                    mode = MODE[controller]\n                    weights=turn_weights if controller in [\'calibrated_pressure\',\'predictive\'] else None\n                    rows={g:phase_rows(item[\'maps\'][g],obs,capacities,message,weights) for g in item[\'greens\']}\n                    if controller in [\'message_only\',\'moving_only\']:\n                        rows={g:[(q,d,f,a if controller==\'message_only\' else 0,m if controller==\'moving_only\' else 0)\n                                 for q,d,f,a,m in data] for g,data in rows.items()}\n                    if controller==\'predictive\':\n                        scores={g:kernel.predictive(data,0 if g==phase else 5) for g,data in rows.items()}\n                    else:scores={g:kernel.score(mode,data) for g,data in rows.items()}\n                    others=[g for g in item[\'greens\'] if g!=phase]\n                    target=max(others,key=lambda g:(scores[g],t-st[\'last_end\'][g],-g))\n                    overdue=[g for g in others if t-st[\'last_end\'][g]>=SETTINGS[\'maximum_red_target_s\']]\n                    if overdue:\n                        target=max(overdue,key=lambda g:(t-st[\'last_end\'][g],-g))\n                        change=elapsed>=SETTINGS[\'minimum_green_s\']\n                    else:\n                        change=kernel.switch(scores[phase],scores[target],elapsed)\n                if change:\n                    if phase in item[\'greens\']:\n                        assert elapsed >= SETTINGS[\'minimum_green_s\']\n                        assert elapsed <= SETTINGS[\'maximum_green_s\']\n                    else:\n                        assert elapsed == p[\'duration\']\n                    if phase in item[\'greens\']:\n                        if controller==\'fixed\':target=item[\'greens\'][(item[\'greens\'].index(phase)+1)%len(item[\'greens\'])]\n                        updates.append((tid,phase+1,elapsed,target))\n                    elif set(p[\'state\'])=={\'r\'}:\n                        assert st[\'target\'] in item[\'greens\']\n                        updates.append((tid,st[\'target\'],elapsed,None))\n                    else:\n                        updates.append((tid,phase+1,elapsed,None))\n            # All agents decide from the same pre-action snapshot.\n            for tid,new,elapsed,target in updates:\n                old = states[tid][\'phase\']\n                sim.trafficlight.setPhase(tid,new)\n                sim.trafficlight.setPhaseDuration(tid,1e7)\n                if old in tls[tid][\'greens\']:\n                    states[tid][\'last_end\'][old]=float(t)\n                    states[tid][\'target\']=target\n                states[tid].update(phase=new,start=float(t))\n                phases_trace.append([t,tid,old,new,elapsed])\n                switches += int(new in tls[tid][\'greens\'])\n            sim.simulationStep()\n            now = float(t+1)\n            # SUMO labels step events by the interval\'s starting time (t),\n            # whereas simulation.getTime() after stepping is t+1.\n            for vid in sim.simulation.getDepartedIDList(): departed[vid] = float(t)\n            for vid in sim.simulation.getArrivedIDList(): arrived[vid] = float(t)\n            collisions += sim.simulation.getCollidingVehiclesNumber()\n            teleports += sim.simulation.getStartingTeleportNumber()\n            if live_hook:live_hook(\'after\',t+1,states,lanes,obs,\n                {\'departed\':departed,\'arrived\':arrived,\'schedule\':schedule,\'collisions\':collisions,\'teleports\':teleports,\'tls\':tls})\n            if (t+1)%5 == 0:\n                requested = sum(d<=now for d in schedule.values())\n                pending = requested-len(departed)\n                assert pending >= 0\n                halted = sum(sim.lane.getLastStepHaltingNumber(l) for l in lanes)\n                timeline.append([now,halted,pending,sim.vehicle.getIDCount(),len(arrived)])\n            if trace and (t+1)%5 == 0:\n                # Compact replay positions are measured SUMO output, not generated imagery.\n                vehicles = [[v,*sim.vehicle.getPosition(v),sim.vehicle.getAngle(v)] for v in sim.vehicle.getIDList()]\n                replay.append({\'t\':now,\'vehicles\':vehicles,\'phases\':{k:v[\'phase\'] for k,v in states.items()}})\n            if stop_when_clear and t+1>=demand_s and len(arrived)==len(schedule):break\n        active = set(sim.vehicle.getIDList())\n        pending_ids = set(schedule)-set(departed)\n        assert set(schedule) == set(arrived) | active | pending_ids\n        assert not (set(arrived)&active or active&pending_ids or set(arrived)&pending_ids)\n    finally:\n        sim.close()\n    trips = ET.parse(out/\'tripinfo.xml\').getroot().findall(\'tripinfo\')\n    inserted = [x for x in trips if float(x.attrib[\'depart\']) >= 0]\n    assert {x.attrib[\'id\'] for x in inserted} == set(departed), \'Trip ledger missing inserted vehicles\'\n    # Count every requested vehicle, including requests never inserted.\n    system_time = sum(max(0.0,arrived.get(v,horizon)-d) for v,d in schedule.items())\n    # Independent check against SUMO\'s XML ledger, not the online arrival map.\n    xml_arrivals = {x.attrib[\'id\']:float(x.attrib[\'arrival\']) for x in inserted\n                    if float(x.attrib[\'arrival\']) >= 0}\n    ledger_time = sum(xml_arrivals.get(v,horizon)-d for v,d in schedule.items())\n    assert math.isclose(system_time,ledger_time,abs_tol=1e-6), \'Arrival-clock mismatch\'\n    insertion_wait = sum(max(0.0,float(x.attrib[\'depart\'])-schedule[x.attrib[\'id\']]) for x in inserted)\n    insertion_wait += sum(horizon-schedule[v] for v in pending_ids)\n    time_loss = sum(float(x.attrib[\'timeLoss\']) for x in inserted)\n    completed_wait = [float(x.attrib[\'waitingTime\']) for x in inserted if float(x.attrib[\'arrival\']) >= 0]\n    all_wait = [float(x.attrib[\'waitingTime\']) for x in inserted]\n    audit = {\'conservation_pass\': True, \'signal_transition_pass\':True,\'metric_ledger_pass\':True,\n             \'colliding_vehicle_events\':collisions,\'teleport_vehicle_events\':teleports,\n             \'valid_run\':collisions == 0 and teleports == 0,\n             \'detector_stats\':observer.stats}\n    result = {\'controller\':controller,\'scenario\':demand.stem.split(\'_seed\')[0],\n        \'regime\':regime,\'seed\':seed,\'horizon_s\':horizon,\'demand_s\':demand_s,\n        \'fixed_green_s\':fixed_green,\'scheduled\':len(schedule),\'departed\':len(departed),\n        \'finished_at_s\':t+1,\'cleared\':len(arrived)==len(schedule),\n        \'controller_config\':kernel.config if controller==\'predictive\' else {},\n        \'arrived\':len(arrived),\'active\':len(active),\'pending\':len(pending_ids),\n        \'completion_fraction\':len(arrived)/len(schedule) if schedule else 1.0,\n        \'total_system_time_s\':round(system_time,3),\n        \'mean_system_time_s\':round(system_time/max(1,len(schedule)),3),\n        \'network_time_loss_plus_entry_wait_s\':round(time_loss+insertion_wait,3),\n        \'entry_wait_s\':round(insertion_wait,3),\n        \'p95_completed_stopped_wait_s\':round(float(np.percentile(completed_wait,95)),3) if completed_wait else None,\n        \'max_inserted_stopped_wait_s\':max(all_wait,default=0),\n        \'switches\':switches,\'wall_s\':round(time.monotonic()-started,3),\n        \'demand_sha256\':sha(demand),\'network_sha256\':sha(netfile), **audit}\n    write_json(out/\'result.json\',result)\n    for name,header,rows in [\n        (\'timeline.csv\',[\'time_s\',\'halted\',\'pending\',\'active\',\'arrived\'],timeline),\n        (\'signals.csv\',[\'time_s\',\'tls\',\'old_phase\',\'new_phase\',\'old_duration_s\'],phases_trace)]:\n        with (out/name).open(\'w\',newline=\'\') as f:\n            w=csv.writer(f);w.writerow(header);w.writerows(rows)\n    if replay: write_json(out/\'replay.json\',replay)\n    return result\n\n\ndef summarize(results,output):\n    output.mkdir(exist_ok=True,parents=True)\n    keys = list(results[0])\n    with (output/\'results.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=keys);w.writeheader()\n        w.writerows({k:json.dumps(v) if isinstance(v,dict) else v for k,v in row.items()} for row in results)\n    groups = defaultdict(list)\n    for r in results: groups[(r[\'scenario\'],r[\'regime\'],r[\'controller\'])].append(r)\n    aggregate=[]\n    for (scenario,regime,controller),rows in sorted(groups.items()):\n        refs = {r[\'seed\']:r for r in results if r[\'scenario\']==scenario and r[\'regime\']==regime and r[\'controller\']==\'max_pressure\'}\n        paired = [(refs[r[\'seed\']][\'total_system_time_s\']-r[\'total_system_time_s\'])/refs[r[\'seed\']][\'total_system_time_s\']*100\n                  for r in rows if r[\'seed\'] in refs]\n        aggregate.append({\'scenario\':scenario,\'regime\':regime,\'controller\':controller,\n            \'runs\':len(rows),\'mean_system_time_s\':round(statistics.mean(r[\'mean_system_time_s\'] for r in rows),3),\n            \'mean_completion\':round(statistics.mean(r[\'completion_fraction\'] for r in rows),4),\n            \'mean_paired_improvement_vs_pressure_pct\':round(statistics.mean(paired),3) if paired else None,\n            \'min_paired_improvement_vs_pressure_pct\':round(min(paired),3) if paired else None,\n            \'max_paired_improvement_vs_pressure_pct\':round(max(paired),3) if paired else None,\n            \'all_runs_valid\':all(r[\'valid_run\'] for r in rows)})\n    write_json(output/\'aggregate.json\',aggregate)\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    combinations=sorted({(r[\'scenario\'],r[\'regime\']) for r in results})\n    fig,axes=plt.subplots(len(combinations),1,figsize=(11,3.2*len(combinations)),squeeze=False)\n    colors=[\'#aab3c0\',\'#6197b7\',\'#173a58\',\'#36a79c\',\'#e69c3a\']\n    for ax,(scenario,regime) in zip(axes[:,0],combinations):\n        for j,c in enumerate(CONTROLLERS):\n            rows=groups.get((scenario,regime,c),[])\n            values=[r[\'mean_system_time_s\'] for r in rows]\n            if values:\n                ax.bar(j,statistics.mean(values),color=colors[j],width=0.65)\n                ax.scatter([j+(i-(len(values)-1)/2)*0.08 for i in range(len(values))],values,s=18,c=\'#132338\',zorder=3)\n        ax.set_xticks(range(5),[\'Fixed\\n(dev selected)\',\'Queue\\nactuated\',\'Max\\npressure\',\'Receiving\\naware\',\'Coordinated\'])\n        ax.set_ylabel(\'System time / request (s)\\nLower is better\')\n        ax.set_title(f\'{scenario.replace("_"," ").title()} | {regime} observations\',loc=\'left\',fontweight=\'bold\')\n        ax.spines[[\'top\',\'right\']].set_visible(False)\n        ax.grid(axis=\'y\',alpha=0.15);ax.set_axisbelow(True)\n    fig.suptitle(\'Adaptive Signals — Stage 1 synthetic-network pilot\\nDots are seeds; unfinished and uninserted vehicles remain in the metric\',fontsize=13)\n    fig.tight_layout(rect=(0,0,1,0.95))\n    fig.savefig(output/\'pilot_comparison.png\',dpi=160)\n    plt.close(fig)\n    return aggregate\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--profile\',choices=[\'smoke\',\'pilot\',\'extended\'],default=\'pilot\')\n    p.add_argument(\'--output\',type=Path,default=ROOT/\'runs/pilot\')\n    p.add_argument(\'--seeds\',type=int,nargs=\'+\')\n    p.add_argument(\'--demand-s\',type=int)\n    p.add_argument(\'--drain-s\',type=int)\n    p.add_argument(\'--resume\',action=\'store_true\')\n    p.add_argument(\'--fixed-green\',type=int,default=None,help=\'Override dev selection (10..50 s)\')\n    args=p.parse_args()\n    demand_s=args.demand_s or (120 if args.profile==\'smoke\' else 900)\n    drain_s=args.drain_s if args.drain_s is not None else (120 if args.profile==\'smoke\' else 600)\n    if demand_s<1 or drain_s<0:p.error(\'Invalid demand/drain duration\')\n    if args.fixed_green is not None and not 10<=args.fixed_green<=50:p.error(\'Fixed green must be 10..50\')\n    seeds=args.seeds or ([777] if args.profile==\'smoke\' else SETTINGS[\'evaluation_seeds\'])\n    if len(set(seeds))!=len(seeds):p.error(\'Seeds must be unique\')\n    if SETTINGS[\'development_seed\'] in seeds:p.error(\'Development seed cannot be an evaluation seed\')\n    scenarios=[\'balanced\'] if args.profile==\'smoke\' else SCENARIOS\n    output=args.output.resolve();output.mkdir(parents=True,exist_ok=True)\n    source_hashes={str(q.relative_to(ROOT)):sha(q) for q in [ROOT/\'experiment.py\',ROOT/\'src/controllers.cpp\',ROOT/\'requirements.txt\']}\n    signature={\'settings\':SETTINGS,\'source_sha256\':source_hashes,\'demand_s\':demand_s,\'drain_s\':drain_s,\'seeds\':seeds,\n        \'scenarios\':scenarios,\'profile\':args.profile,\'fixed_override\':args.fixed_green,\n        \'sumo_version\':sim.getVersion()[1]}\n    manifest_path=output/\'manifest.json\'\n    if manifest_path.exists():\n        old=json.loads(manifest_path.read_text())\n        if old[\'signature\']!=signature:raise RuntimeError(\'Output belongs to a different configuration/source version. Use a new --output.\')\n        if not args.resume:raise RuntimeError(\'Output already exists; use --resume or a new --output\')\n    else:\n        write_json(manifest_path,{\'signature\':signature,\'python\':sys.version,\'platform\':platform.platform(),\n            \'started_utc\':time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()),\n            \'metric\':\'sum(min(arrival,horizon)-requested_depart) over every requested vehicle\',\n            \'limitations\':[\'synthetic grid and demand\',\'no RL\',\'no CCTV\',\'no pedestrians/motorcycles\',\n                \'equal turning weights\',\'three-seed pilot is not a statistical superiority claim\']})\n    netfile,net,tls=build_network(output/\'inputs\')\n    def cached_run(folder,*params,**kwargs):\n        file=folder/\'result.json\'\n        if args.resume and file.exists():\n            r=json.loads(file.read_text())\n            # Input identity and controller configuration are verified on resume.\n            assert r[\'network_sha256\']==sha(netfile) and r[\'demand_sha256\']==sha(params[0])\n            return r\n        return run_one(folder,netfile,tls,*params,**kwargs)\n    # Hold-out development stream only: select fixed duration on identical scenarios.\n    if args.fixed_green is None:\n        tuning=[]\n        for scenario in scenarios:\n            d,s=generate_demand(output/\'inputs\',net,scenario,101,demand_s)\n            for green in SETTINGS[\'fixed_candidates_s\']:\n                row=cached_run(output/\'development\'/f\'{scenario}_g{green}\',d,s,\'fixed\',\'perfect\',101,demand_s,drain_s,green,False)\n                tuning.append(row)\n        fixed=min(SETTINGS[\'fixed_candidates_s\'],key=lambda g:statistics.mean(r[\'mean_system_time_s\'] for r in tuning if r[\'fixed_green_s\']==g))\n        write_json(output/\'fixed_selection.json\',{\'selected_green_s\':fixed,\'development_results\':tuning})\n    else:fixed=args.fixed_green\n    campaign=[]\n    for scenario in scenarios:\n        for regime in [\'perfect\',\'degraded\']:\n            # Pilot degrades the most transient scenario; extended degrades all.\n            if args.profile==\'pilot\' and regime==\'degraded\' and scenario!=\'directional_switch\':continue\n            for seed in seeds:\n                d,s=generate_demand(output/\'inputs\',net,scenario,seed,demand_s)\n                for c in CONTROLLERS:\n                    dest=output/\'evaluation\'/f\'{scenario}_{regime}_seed{seed}_{c}\'\n                    row=cached_run(dest,d,s,c,regime,seed,demand_s,drain_s,fixed)\n                    campaign.append(row)\n                    write_json(output/\'progress.json\',{\'completed\':len(campaign),\'last\':row})\n                    print(f\'{len(campaign):03d} {scenario:18s} {regime:8s} seed={seed} {c:16s} \'\n                          f\'system={row["mean_system_time_s"]:.1f}s arrived={row["arrived"]}/{row["scheduled"]} \'\n                          f\'valid={row["valid_run"]}\',flush=True)\n    aggregate=summarize(campaign,output)\n    assert all(r[\'valid_run\'] for r in campaign),\'Invalid runs retained in results; investigate SUMO logs\'\n    print(f\'Completed {len(campaign)} evaluation runs. Results: {output}\',flush=True)\n    return aggregate\n\n\nif __name__==\'__main__\': main()\n', 'stage2.py': '"""Stage-2 development selection and untouched-seed evaluation."""\nfrom __future__ import annotations\nimport argparse\nfrom collections import defaultdict\nimport csv\nimport json\nfrom pathlib import Path\nimport statistics\nimport sys\nimport time\nimport xml.etree.ElementTree as ET\nimport experiment as e\n\nCONTROLLERS=[\'fixed\',\'sumo_actuated\',\'max_pressure\',\'calibrated_pressure\',\n             \'receiving_aware\',\'message_only\',\'moving_only\',\'coordinated\',\'predictive\']\nLOADS=[0.3,0.6,1.0]\nDEV_SEEDS=[201,1201]\nEVAL_SEEDS=[12777,13777,14777,15777]\nCONFIGS=[{\'horizon_s\':h,\'receiving_power\':p,\'message_weight\':0.3,\'moving_weight\':0.2}\n         for h,p in [(10,1),(20,1),(30,1),(20,2)]]\nDEV_CASES=[(\'balanced\',0.3),(\'directional_switch\',0.6),(\'surge\',1.0)]\n\n\ndef learn_turns(demands):\n    counts=defaultdict(lambda:defaultdict(int))\n    for path in demands:\n        for v in ET.parse(path).getroot().findall(\'vehicle\'):\n            edges=v.find(\'route\').attrib[\'edges\'].split()\n            for incoming,outgoing in zip(edges,edges[1:]):\n                counts[incoming+\'_0\'][outgoing+\'_0\']+=1\n    return {lane:{out:n/sum(outs.values()) for out,n in sorted(outs.items())}\n            for lane,outs in sorted(counts.items())}\n\n\ndef aggregate(rows,folder):\n    with (folder/\'results.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader()\n        w.writerows({k:json.dumps(v) if isinstance(v,dict) else v for k,v in r.items()} for r in rows)\n    groups=defaultdict(list)\n    for r in rows:groups[(r[\'scenario\'],r[\'load_scale\'],r[\'regime\'],r[\'controller\'])].append(r)\n    output=[]\n    for (scenario,load,regime,controller),g in sorted(groups.items()):\n        refs={r[\'seed\']:r for r in rows if r[\'scenario\']==scenario and r[\'load_scale\']==load and r[\'regime\']==regime and r[\'controller\']==\'fixed\'}\n        change=[100*(refs[r[\'seed\']][\'total_system_time_s\']-r[\'total_system_time_s\'])/refs[r[\'seed\']][\'total_system_time_s\'] for r in g]\n        output.append({\'scenario\':scenario,\'load_scale\':load,\'regime\':regime,\'controller\':controller,\'runs\':len(g),\n            \'mean_system_time_s\':statistics.mean(r[\'mean_system_time_s\'] for r in g),\n            \'mean_completion\':statistics.mean(r[\'completion_fraction\'] for r in g),\n            \'cleared_runs\':sum(r[\'cleared\'] for r in g),\n            \'paired_reduction_vs_fixed_pct\':statistics.mean(change),\n            \'min_reduction_pct\':min(change),\'max_reduction_pct\':max(change),\n            \'mean_max_stopped_wait_s\':statistics.mean(r[\'max_inserted_stopped_wait_s\'] for r in g),\n            \'mean_switches\':statistics.mean(r[\'switches\'] for r in g),\n            \'all_valid\':all(r[\'valid_run\'] for r in g)})\n    e.write_json(folder/\'aggregate.json\',output)\n    plot(output,folder)\n    return output\n\n\ndef plot(agg,folder):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    chosen=[\'fixed\',\'sumo_actuated\',\'calibrated_pressure\',\'coordinated\',\'predictive\']\n    labels=[\'Fixed\',\'SUMO actuated\',\'Calibrated pressure\',\'Stage-1 coordinated\',\'Discharge lookahead\']\n    colors=[\'#8e9caf\',\'#5278ab\',\'#173a58\',\'#35a89a\',\'#e7a140\']\n    fig,axes=plt.subplots(2,3,figsize=(15,8),sharey=False)\n    for i,scenario in enumerate([\'balanced\',\'directional_switch\']):\n        for j,load in enumerate(LOADS):\n            ax=axes[i,j]\n            data=[next((r for r in agg if r[\'scenario\']==scenario and r[\'load_scale\']==load and r[\'regime\']==\'perfect\' and r[\'controller\']==c),None) for c in chosen]\n            for k,r in enumerate(data):\n                if r:ax.bar(k,r[\'mean_system_time_s\'],color=colors[k]);ax.text(k,r[\'mean_system_time_s\']+8,f"{r[\'mean_completion\']:.0%}",ha=\'center\',fontsize=8)\n            ax.set_title(f\'{scenario.replace("_"," ").title()} | load ×{load:g}\',loc=\'left\',fontsize=11,fontweight=\'bold\')\n            ax.set_xticks(range(len(chosen)),[\'Fixed\',\'SUMO\\nactuated\',\'Turn\\npressure\',\'Stage-1\\ncoord.\',\'Discharge\\nlookahead\'],fontsize=9)\n            ax.set_ylim(0,max((r[\'mean_system_time_s\'] for r in data if r),default=1)*1.2)\n            ax.spines[[\'top\',\'right\']].set_visible(False);ax.grid(axis=\'y\',alpha=.15);ax.set_axisbelow(True)\n            if j==0:ax.set_ylabel(\'System time / request (s)\')\n    fig.suptitle(\'Stage 2 — broader demand and longer draining\\nLabels show completion fraction; four fresh seeds; synthetic network\',fontsize=14)\n    fig.tight_layout(rect=[0,0,1,.92]);fig.savefig(folder/\'stage2_comparison.png\',dpi=150);plt.close(fig)\n\n\ndef campaign(folder,profile=\'full\',resume=False):\n    folder=Path(folder).resolve();folder.mkdir(parents=True,exist_ok=True)\n    demand_s,drain_s=(120,300) if profile==\'smoke\' else (900,1800)\n    seeds=EVAL_SEEDS[:1] if profile==\'smoke\' else EVAL_SEEDS\n    source={str(p.relative_to(e.ROOT)):e.sha(p) for p in [e.ROOT/\'experiment.py\',e.ROOT/\'stage2.py\',e.ROOT/\'src/controllers.cpp\']}\n    signature={\'source_sha256\':source,\'settings\':e.SETTINGS,\'dev_cases\':DEV_CASES,\'development_seeds\':DEV_SEEDS,\n        \'evaluation_seeds\':seeds,\'configs\':CONFIGS,\'fixed_candidates\':[10,20,30,45],\n        \'profile\':profile,\'demand_s\':demand_s,\'drain_s\':drain_s,\'sumo_version\':e.sim.getVersion()[1]}\n    # Normalize tuples exactly as JSON writes them for idempotent resume.\n    signature=json.loads(json.dumps(signature))\n    manifest=folder/\'manifest.json\'\n    if manifest.exists():\n        assert json.loads(manifest.read_text())[\'signature\']==signature,\'Use a new output folder after source/config changes\'\n        if not resume:raise RuntimeError(\'Use --resume or a new output folder\')\n    else:e.write_json(manifest,{\'signature\':signature,\'started_utc\':time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()),\n        \'plan\':\'development-only tuning, frozen before fresh-seed evaluation; 900 s demand / 1800 s max drain\',\n        \'limits\':[\'synthetic demand/network\',\'no RL\',\'no live CCTV\',\'approximate discharge and pressure models\']})\n    nf,net,tls=e.build_network(folder/\'inputs\')\n    dev=[]\n    for scenario,load in DEV_CASES:\n        for seed in DEV_SEEDS:\n            d,s=e.generate_demand(folder/\'inputs\',net,scenario,seed,demand_s,load)\n            dev.append((scenario,load,seed,d,s))\n    turns=learn_turns([d for _,_,_,d,_ in dev]);e.write_json(folder/\'turn_weights.json\',turns)\n    def run(dest,scenario,load,seed,d,s,c,regime=\'perfect\',green=20,config=None,trace=False):\n        result=dest/\'result.json\'\n        if resume and result.exists():\n            r=json.loads(result.read_text());assert r[\'demand_sha256\']==e.sha(d) and r[\'network_sha256\']==e.sha(nf)\n            return r\n        r=e.run_one(dest,nf,tls,d,s,c,regime,seed,demand_s,drain_s,green,trace,\n            config=config,turn_weights=turns,stop_when_clear=True)\n        r.update(load_scale=load)\n        e.write_json(result,r)\n        if not r[\'valid_run\']:raise RuntimeError(f\'Invalid run retained: {dest}\')\n        return r\n    fixed_results=[]\n    for scenario,load,seed,d,s in dev:\n        for green in [10,20,30,45]:\n            dest=folder/\'development\'/f\'{scenario}_l{load}_seed{seed}_fixed_g{green}\'\n            fixed_results.append(run(dest,scenario,load,seed,d,s,\'fixed\',green=green))\n    fixed={str(load):min([10,20,30,45],key=lambda g:statistics.mean(r[\'mean_system_time_s\'] for r in fixed_results if r[\'load_scale\']==load and r[\'fixed_green_s\']==g)) for load in LOADS}\n    scores=[];tuning=[]\n    for ci,config in enumerate(CONFIGS):\n        ratios=[]\n        for scenario,load,seed,d,s in dev:\n            dest=folder/\'development\'/f\'{scenario}_l{load}_seed{seed}_predictive_c{ci}\'\n            r=run(dest,scenario,load,seed,d,s,\'predictive\',green=fixed[str(load)],config=config)\n            baseline=next(x for x in fixed_results if x[\'scenario\']==scenario and x[\'load_scale\']==load and x[\'seed\']==seed and x[\'fixed_green_s\']==fixed[str(load)])\n            ratios.append(r[\'total_system_time_s\']/baseline[\'total_system_time_s\']);tuning.append(r)\n        scores.append(statistics.mean(ratios))\n        print(f\'DEV candidate={ci} relative_system_time={scores[-1]:.4f}\',flush=True)\n    chosen=min(range(len(CONFIGS)),key=lambda i:scores[i]);config=CONFIGS[chosen]\n    selection={\'fixed_green_by_load\':fixed,\'predictive_config\':config,\'candidate_scores\':scores,\n        \'objective\':\'mean ratio of total system time to dev-selected fixed timing across six dev streams\',\n        \'selected_before_evaluation\':True,\'turn_weight_sha256\':e.sha(folder/\'turn_weights.json\')}\n    e.write_json(folder/\'selection.json\',selection)\n    print(\'SELECTION FROZEN\',json.dumps(selection),flush=True)\n    cases=[(scenario,load,\'perfect\') for scenario in [\'balanced\',\'directional_switch\'] for load in LOADS]\n    cases += [(\'surge\',0.6,\'perfect\'),(\'directional_switch\',1.0,\'degraded\')]\n    if profile==\'smoke\':cases=[(\'balanced\',0.3,\'perfect\')]\n    results=[]\n    for scenario,load,regime in cases:\n        for seed in seeds:\n            d,s=e.generate_demand(folder/\'inputs\',net,scenario,seed,demand_s,load)\n            for c in CONTROLLERS:\n                # Built-in actuated uses its own loop detectors, not the camera model.\n                if regime==\'degraded\' and c==\'sumo_actuated\':continue\n                dest=folder/\'evaluation\'/f\'{scenario}_l{load}_{regime}_seed{seed}_{c}\'\n                r=run(dest,scenario,load,seed,d,s,c,regime=regime,green=fixed[str(load)],config=config)\n                results.append(r)\n                e.write_json(folder/\'progress.json\',{\'completed\':len(results),\'last\':r})\n                print(f"EVAL {len(results):03d} {scenario:18s} l={load:g} {regime:8s} seed={seed} {c:19s} "\n                      f"system={r[\'mean_system_time_s\']:.1f}s complete={r[\'completion_fraction\']:.1%} valid={r[\'valid_run\']}",flush=True)\n    agg=aggregate(results,folder)\n    expected=9 if profile==\'smoke\' else 284\n    assert len(results)==expected\n    groups=defaultdict(list)\n    for r in results:groups[(r[\'scenario\'],r[\'load_scale\'],r[\'regime\'],r[\'seed\'])].append(r)\n    for g in groups.values():\n        assert len({r[\'demand_sha256\'] for r in g})==len({r[\'network_sha256\'] for r in g})==1\n    e.write_json(folder/\'audit.json\',{\'evaluation_runs\':len(results),\'development_runs\':len(fixed_results)+len(tuning),\n        \'all_valid\':all(r[\'valid_run\'] for r in results),\'matched_input_groups\':len(groups),\n        \'collision_events\':sum(r[\'colliding_vehicle_events\'] for r in results),\'teleport_events\':sum(r[\'teleport_vehicle_events\'] for r in results),\n        \'all_metric_ledgers_agree\':all(r[\'metric_ledger_pass\'] for r in results),\'selected_before_evaluation\':True})\n    print(f\'CAMPAIGN_COMPLETE {len(results)} evaluation runs\',flush=True)\n    return agg\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--profile\',choices=[\'smoke\',\'full\'],default=\'full\')\n    p.add_argument(\'--output\',type=Path,default=e.ROOT/\'runs/stage2\')\n    p.add_argument(\'--resume\',action=\'store_true\')\n    a=p.parse_args();campaign(a.output,a.profile,a.resume)\n', 'live_server.py': '"""Local/Colab live SUMO viewer. One simulation-owning worker, HTTP snapshots.\n\nThe browser draws actual SUMO positions; it does not simulate vehicle motion.\nNo cloud hosting is implied. Bind only to loopback and use Colab\'s private port\nproxy or a local browser. Ctrl+C closes the server and simulation worker.\n"""\nfrom __future__ import annotations\nimport argparse\nimport copy\nimport json\nimport queue\nimport secrets\nimport signal\nimport threading\nimport time\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\nimport urllib.parse\nimport experiment as e\nfrom stage2 import CONTROLLERS, learn_turns\n\n\nclass CancelRun(Exception):pass\n\n\nclass LiveApp:\n    def __init__(self,folder):\n        self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True)\n        self.netfile,self.net,self.tls=e.build_network(self.folder/\'inputs\')\n        self.cv=threading.Condition();self.commands=queue.Queue()\n        self.paused=False;self.step_budget=0;self.cancel=False;self.closed=False;self.speed=10.0\n        self.snapshot={\'status\':\'ready\',\'time_s\':0,\'vehicles\':[],\'signals\':[],\n            \'metrics\':{},\'history\':[],\'message\':\'Choose a controller and start a live run.\'}\n        self.scene={\'bounds\':self.net.getBoundary(),\'lanes\':[\n            {\'id\':l.getID(),\'shape\':l.getShape(),\'length\':l.getLength(),\n             \'from\':edge.getFromNode().getID(),\'to\':edge.getToNode().getID()}\n            for edge in self.net.getEdges() for l in edge.getLanes()],\n            \'nodes\':[{\'id\':tid,\'xy\':self.net.getNode(tid).getCoord()} for tid in self.tls]}\n        self.config={\'controller\':\'predictive\',\'scenario\':\'directional_switch\',\'scale\':0.6,\n                     \'regime\':\'perfect\',\'seed\':12777,\'demand_s\':600}\n        selection=e.ROOT/\'runs/stage2/selection.json\'\n        self.selection=json.loads(selection.read_text()) if selection.exists() else {\n            \'predictive_config\':{\'horizon_s\':20,\'receiving_power\':1,\'message_weight\':0.3,\'moving_weight\':0.2},\n            \'fixed_green_by_load\':{\'0.3\':10,\'0.6\':10,\'1.0\':20}}\n        self.turns=None\n        weights=e.ROOT/\'runs/stage2/turn_weights.json\'\n        if weights.exists():self.turns=json.loads(weights.read_text())\n        self.worker=threading.Thread(target=self._worker,name=\'sumo-owner\',daemon=True);self.worker.start()\n\n    def control(self,action,values=None):\n        values=values or {}\n        with self.cv:\n            if action==\'start\':\n                c={**self.config,**values}\n                if c[\'controller\'] not in CONTROLLERS:raise ValueError(\'Unknown controller\')\n                if c[\'scenario\'] not in e.SCENARIOS:raise ValueError(\'Unknown traffic pattern\')\n                if c[\'regime\'] not in [\'perfect\',\'degraded\']:raise ValueError(\'Unknown sensor condition\')\n                if c[\'controller\']==\'sumo_actuated\' and c[\'regime\']==\'degraded\':\n                    raise ValueError(\'SUMO actuated uses loop detectors; select perfect sensors for that reference.\')\n                if not 0.1<=float(c[\'scale\'])<=1.5:raise ValueError(\'Demand multiplier must be 0.1–1.5\')\n                if not 0<=int(c[\'seed\'])<2**31:raise ValueError(\'Seed outside supported range\')\n                if not 60<=int(c[\'demand_s\'])<=3600:raise ValueError(\'Demand duration must be 60–3600 s\')\n                c.update(scale=float(c[\'scale\']),seed=int(c[\'seed\']),demand_s=int(c[\'demand_s\']))\n                self.config=c;self.cancel=True;self.paused=False;self.step_budget=0\n                self.snapshot[\'status\']=\'starting\';self.commands.put(c)\n            elif action==\'pause\':self.paused=True\n            elif action==\'resume\':self.paused=False\n            elif action==\'step\':self.paused=True;self.step_budget+=1\n            elif action==\'speed\':\n                speed=float(values.get(\'speed\',10))\n                if not 1<=speed<=60:raise ValueError(\'Speed must be 1–60×\')\n                self.speed=speed\n            elif action==\'stop\':self.cancel=True;self.paused=False\n            else:raise ValueError(\'Unknown action\')\n            self.cv.notify_all()\n\n    def state(self):\n        with self.cv:\n            data=copy.deepcopy(self.snapshot)\n            if data[\'status\']==\'running\' and self.paused:data[\'status\']=\'paused\'\n            data.update(speed=self.speed,config=self.config)\n            return data\n\n    def _worker(self):\n        while not self.closed:\n            c=self.commands.get()\n            if c is None:return\n            # Newer reset commands supersede queued, unstarted requests.\n            while not self.commands.empty():\n                later=self.commands.get_nowait()\n                if later is None:return\n                c=later\n            with self.cv:self.cancel=False\n            try:\n                d,s=e.generate_demand(self.folder/\'inputs\',self.net,c[\'scenario\'],c[\'seed\'],c[\'demand_s\'],c[\'scale\'])\n                turns=self.turns\n                if turns is None:\n                    # Default calibration is a development stream, never this live stream.\n                    dd,_=e.generate_demand(self.folder/\'inputs\',self.net,\'balanced\',201,900,0.6)\n                    turns=learn_turns([dd])\n                self.history=[]\n                with self.cv:self.snapshot.update(status=\'running\',message=\'\',history=[],metrics={},time_s=0,vehicles=[],signals=[])\n                closest=min(self.selection[\'fixed_green_by_load\'],key=lambda x:abs(float(x)-c[\'scale\']))\n                green=self.selection[\'fixed_green_by_load\'][closest]\n                result=e.run_one(self.folder/\'latest\',self.netfile,self.tls,d,s,c[\'controller\'],c[\'regime\'],\n                    c[\'seed\'],c[\'demand_s\'],1800,green,False,config=self.selection[\'predictive_config\'],\n                    turn_weights=turns,stop_when_clear=True,live_hook=self._hook)\n                with self.cv:self.snapshot.update(status=\'completed\',result=result,\n                    message=\'All requests cleared.\' if result[\'cleared\'] else \'Cutoff reached; unfinished vehicles remain counted.\')\n            except CancelRun:\n                with self.cv:self.snapshot.update(status=\'stopped\',message=\'Run stopped. Start resets the same seeded experiment.\')\n            except Exception as exc:\n                with self.cv:self.snapshot.update(status=\'error\',message=f\'{type(exc).__name__}: {exc}\')\n\n    def _hook(self,event,t,states,lanes,obs,ledger):\n        if event==\'before\':\n            with self.cv:\n                while self.paused and self.step_budget==0 and not self.cancel and not self.closed:self.cv.wait()\n                if self.cancel or self.closed:raise CancelRun()\n                if self.step_budget:self.step_budget-=1\n                # Event-based pacing keeps pause/reset responsive; no busy loop.\n                self.cv.wait(timeout=1.0/self.speed)\n                if self.cancel or self.closed:raise CancelRun()\n            return\n        vehicles=[]\n        for v in e.sim.vehicle.getIDList():\n            x,y=e.sim.vehicle.getPosition(v)\n            vehicles.append([v,round(x,2),round(y,2),round(e.sim.vehicle.getAngle(v),1),round(e.sim.vehicle.getSpeed(v),2)])\n        signals=[]\n        for tid,item in ledger[\'tls\'].items():\n            phase=e.sim.trafficlight.getPhase(tid)\n            for g,mapping in item[\'maps\'].items():\n                color=\'G\' if phase==g else \'y\' if phase==g+1 else \'r\'\n                for lane in mapping:signals.append([lane,color])\n        requested={v:d for v,d in ledger[\'schedule\'].items() if d<=t}\n        pending=len(requested)-len(ledger[\'departed\'])\n        halted=sum(e.sim.lane.getLastStepHaltingNumber(lane) for lane in lanes)\n        system=sum(max(0,ledger[\'arrived\'].get(v,t)-d) for v,d in requested.items())\n        metrics={\'requested\':len(requested),\'total_requests\':len(ledger[\'schedule\']),\n            \'arrived\':len(ledger[\'arrived\']),\'active\':len(vehicles),\'pending\':pending,\'halted\':halted,\n            \'mean_system_so_far_s\':round(system/max(1,len(requested)),1),\n            \'collisions\':ledger[\'collisions\'],\'teleports\':ledger[\'teleports\']}\n        if t%5==0:self.history.append([t,halted,pending,len(ledger[\'arrived\'])])\n        sensors={l:[round(q,1),round(n,1)] for l,(q,n) in obs.items()}\n        with self.cv:self.snapshot.update(time_s=t,vehicles=vehicles,signals=signals,metrics=metrics,\n            history=self.history[-600:],sensors=sensors,status=\'running\')\n\n    def close(self):\n        with self.cv:self.closed=True;self.cancel=True;self.paused=False;self.cv.notify_all()\n        self.commands.put(None);self.worker.join(timeout=5)\n\n\ndef server(app,host=\'127.0.0.1\',port=8765):\n    access=secrets.token_urlsafe(24)\n    class Handler(BaseHTTPRequestHandler):\n        def log_message(self,*args):pass\n        def send(self,code,payload,kind=\'application/json\'):\n            data=payload.encode() if isinstance(payload,str) else json.dumps(payload,allow_nan=False).encode()\n            self.send_response(code);self.send_header(\'Content-Type\',kind+\'; charset=utf-8\')\n            self.send_header(\'Cache-Control\',\'no-store\');self.send_header(\'Content-Length\',str(len(data)))\n            self.end_headers();self.wfile.write(data)\n        def do_GET(self):\n            path=urllib.parse.urlparse(self.path).path\n            if path.endswith(\'/api/state\') or path.endswith(\'/api/scene\'):\n                if self.headers.get(\'X-Simulation-Key\')!=access:return self.send(403,{\'error\':\'Invalid simulation key\'})\n                return self.send(200,app.state() if path.endswith(\'/api/state\') else app.scene)\n            if path.endswith(\'/\') or path.endswith(\'/index.html\'):\n                html=(e.ROOT/\'viewer/index.html\').read_text().replace(\'__ACCESS_KEY__\',json.dumps(access))\n                return self.send(200,html,\'text/html\')\n            self.send(404,{\'error\':\'Not found\'})\n        def do_POST(self):\n            if not urllib.parse.urlparse(self.path).path.endswith(\'/api/control\'):return self.send(404,{\'error\':\'Not found\'})\n            if self.headers.get(\'X-Simulation-Key\')!=access:return self.send(403,{\'error\':\'Invalid simulation key\'})\n            length=int(self.headers.get(\'Content-Length\',\'0\'))\n            if not 0<length<=8192:return self.send(400,{\'error\':\'Invalid request length\'})\n            try:\n                body=json.loads(self.rfile.read(length));app.control(body[\'action\'],body.get(\'values\'))\n                self.send(200,{\'ok\':True})\n            except (KeyError,TypeError,ValueError) as exc:self.send(400,{\'error\':str(exc)})\n    http=ThreadingHTTPServer((host,port),Handler)\n    http.daemon_threads=True\n    return http\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--port\',type=int,default=8765)\n    p.add_argument(\'--output\',type=Path,default=e.ROOT/\'runs/live\')\n    a=p.parse_args();app=LiveApp(a.output);http=server(app,port=a.port)\n    def interrupt(*_args):raise KeyboardInterrupt()\n    signal.signal(signal.SIGTERM,interrupt)\n    print(f\'LIVE_SIMULATION_READY http://127.0.0.1:{a.port}/\',flush=True)\n    try:http.serve_forever(poll_interval=.2)\n    except KeyboardInterrupt:pass\n    finally:http.server_close();app.close()\n', 'make_demo.py': '"""Export a recorded SUMO comparison GIF; this is not the live viewer."""\nimport json\nfrom pathlib import Path\nimport xml.etree.ElementTree as ET\nimport numpy as np\nimport experiment as e\n\n\ndef main():\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    from matplotlib.animation import FuncAnimation,PillowWriter\n    folder=e.ROOT/\'runs/demo\';folder.mkdir(parents=True,exist_ok=True)\n    nf,net,tls=e.build_network(folder/\'inputs\')\n    d,s=e.generate_demand(folder/\'inputs\',net,\'directional_switch\',22777,300,1)\n    selection=json.loads((e.ROOT/\'runs/stage2/selection.json\').read_text())\n    turns=json.loads((e.ROOT/\'runs/stage2/turn_weights.json\').read_text())\n    names=[\'fixed\',\'coordinated\',\'predictive\'];replays=[]\n    for c in names:\n        out=folder/c\n        r=e.run_one(out,nf,tls,d,s,c,\'perfect\',22777,300,300,selection[\'fixed_green_by_load\'][\'1.0\'],\n                    True,config=selection[\'predictive_config\'],turn_weights=turns)\n        assert r[\'valid_run\']\n        replays.append(json.loads((out/\'replay.json\').read_text()))\n    connections={}\n    for c in ET.parse(nf).getroot().findall(\'connection\'):\n        if \'tl\' in c.attrib:\n            lane=c.attrib[\'from\']+\'_\'+c.attrib[\'fromLane\']\n            connections.setdefault(lane,(c.attrib[\'tl\'],int(c.attrib[\'linkIndex\'])))\n    fig,axes=plt.subplots(1,3,figsize=(13,5.5),facecolor=\'#0c1827\')\n    fig.suptitle(\'Same traffic requests. Different signal decisions.\',color=\'#e6eef8\',fontsize=17,y=.96)\n    fig.text(.5,.89,\'Recorded SUMO trajectories · demo seed 22777 · demand ×1.0 · not the live viewer\',ha=\'center\',color=\'#95a9bf\',fontsize=10)\n    scatters=[];labels=[];signal_lines=[]\n    for ax,title in zip(axes,[\'Development-tuned fixed\',\'Stage-1 coordinated\',\'Discharge lookahead\']):\n        ax.set_facecolor(\'#0c1827\');ax.set_aspect(\'equal\');ax.set_xlim(-25,785);ax.set_ylim(-25,785)\n        ax.axis(\'off\');ax.set_title(title,color=\'#e6eef8\',fontsize=11,pad=10)\n        bars={}\n        for edge in net.getEdges():\n            for lane in edge.getLanes():\n                shape=np.array(lane.getShape());ax.plot(shape[:,0],shape[:,1],c=\'#30455d\',lw=3,zorder=1)\n                if lane.getID() in connections:\n                    p,q=shape[-2],shape[-1];delta=(q-p)/np.linalg.norm(q-p);normal=np.array([-delta[1],delta[0]])*4\n                    line,=ax.plot([q[0]-normal[0],q[0]+normal[0]],[q[1]-normal[1],q[1]+normal[1]],lw=2,c=\'#f17b7b\',zorder=3)\n                    bars[lane.getID()]=line\n        for tid in tls:\n            x,y=net.getNode(tid).getCoord();ax.text(x+10,y+13,tid,color=\'#95a9bf\',fontsize=6)\n        scatters.append(ax.scatter([],[],s=7,marker=\'s\',zorder=4))\n        labels.append(ax.text(.5,-.05,\'\',transform=ax.transAxes,color=\'#e6eef8\',fontsize=10,ha=\'center\'))\n        signal_lines.append(bars)\n    clock=fig.text(.5,.045,\'\',ha=\'center\',color=\'#54d5bd\',fontsize=12)\n    fig.subplots_adjust(left=.025,right=.975,top=.81,bottom=.13,wspace=.05)\n    def update(i):\n        for k,replay in enumerate(replays):\n            frame=replay[min(i,len(replay)-1)];vs=frame[\'vehicles\']\n            previous={v[0]:(v[1],v[2]) for v in replay[max(0,i-1)][\'vehicles\']}\n            points=np.array([[v[1],v[2]] for v in vs],dtype=float).reshape((-1,2))\n            colors=[\'#f7ba58\' if v[0] in previous and np.linalg.norm(np.array(previous[v[0]])-np.array([v[1],v[2]]))<1 else \'#54d5bd\' for v in vs]\n            scatters[k].set_offsets(points);scatters[k].set_color(colors)\n            labels[k].set_text(f\'{len(vs)} vehicles in network\')\n            for lane,line in signal_lines[k].items():\n                tid,index=connections[lane];phase=frame[\'phases\'][tid];color=tls[tid][\'phases\'][phase][\'state\'][index]\n                line.set_color(\'#54d5bd\' if color==\'G\' else \'#f7ba58\' if color==\'y\' else \'#f17b7b\')\n        t=replays[0][min(i,len(replays[0])-1)][\'t\'];clock.set_text(f\'Simulation time {int(t)//60:02d}:{int(t)%60:02d}  ·  green: moving / amber: little movement since prior frame\')\n        return scatters+labels+[clock]\n    animation=FuncAnimation(fig,update,frames=len(replays[0]),interval=90,blit=False)\n    animation.save(folder/\'traffic_comparison.gif\',writer=PillowWriter(fps=12),dpi=95)\n    update(min(65,len(replays[0])-1));fig.savefig(folder/\'traffic_snapshot.png\',dpi=150,facecolor=fig.get_facecolor())\n    plt.close(fig)\n    print(folder/\'traffic_comparison.gif\')\n\n\nif __name__==\'__main__\':main()\n', 'src/controllers.cpp': '#include <algorithm>\n#include <cmath>\n#include <limits>\n\n// Stage-1 heuristics, not a claim of a novel max-pressure formulation.\n// Each row is one served incoming lane. Downstream values use equal weights\n// across its allowed outgoing lanes; turning fractions are not observed.\nextern "C" double phase_score(int mode, int n, const double* queue,\n    const double* downstream_queue, const double* downstream_fill,\n    const double* announced, const double* moving) {\n    double result = 0.0;\n    for (int i = 0; i < n; ++i) {\n        double pressure = queue[i] - downstream_queue[i];\n        if (mode == 0) { result += queue[i]; continue; }\n        if (mode == 1) { result += pressure; continue; }\n        double receiving = std::pow(std::clamp(1.0-downstream_fill[i], 0.0, 1.0), 2);\n        if (mode == 3) pressure += 0.30*announced[i] + 0.20*moving[i];\n        result += pressure * receiving;\n    }\n    return result;\n}\n\nextern "C" int should_switch(double current_score, double next_score,\n    double elapsed, double min_green, double max_green, double hysteresis) {\n    if (elapsed < min_green) return 0;\n    if (elapsed >= max_green) return 1;\n    return next_score > current_score + hysteresis;\n}\n\n// Approximate finite-horizon queue-discharge benefit. This is a heuristic,\n// not a calibrated microscopic model or a claim of optimal MPC.\nextern "C" double discharge_score(int n, const double* queue,\n    const double* downstream_queue, const double* downstream_fill,\n    const double* announced, const double* moving, double horizon,\n    double lost_time, double receiving_power, double message_weight,\n    double moving_weight) {\n    const double service_time = std::max(0.0, horizon-lost_time);\n    const double saturation = 0.45; // assumed vehicles/s on one approach\n    double result = 0;\n    for (int i=0;i<n;++i) {\n        const double demand = std::max(0.0,queue[i]+message_weight*announced[i]+moving_weight*moving[i]);\n        const double free_fraction = std::clamp(1-downstream_fill[i],0.0,1.0);\n        const double released = std::min(demand,saturation*service_time);\n        const double wait_saved = released*service_time-released*released/(2*saturation);\n        result += wait_saved*std::pow(free_fraction,receiving_power)\n                  - 0.25*released*downstream_queue[i];\n    }\n    return result;\n}\n', 'requirements.txt': 'eclipse-sumo==1.25.0\nlibsumo==1.25.0\ntraci==1.25.0\nsumolib==1.25.0\nnumpy>=1.26,<3\nmatplotlib>=3.8,<4\n', 'viewer/index.html': '<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Adaptive Signals · Live laboratory</title>\n<style>\n:root{color-scheme:dark;--bg:#09121e;--panel:#101e2e;--line:#25384c;--text:#e6eef8;--muted:#95a9bf;--teal:#54d5bd;--amber:#f7ba58;--red:#f17b7b}*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--text);font:14px system-ui,-apple-system,Segoe UI,sans-serif}header{padding:20px 28px;border-bottom:1px solid var(--line);display:flex;align-items:center;justify-content:space-between;gap:16px}h1{font-size:23px;letter-spacing:-.7px;margin:0}header p{margin:4px 0 0;color:var(--muted);font-size:12px}.brand{display:flex;gap:13px;align-items:center}.mark{display:grid;grid-template-columns:repeat(2,8px);gap:4px;padding:10px;border:1px solid var(--line);border-radius:12px}.mark i{width:8px;height:8px;background:var(--teal);border-radius:2px}.mark i:nth-child(2){opacity:.3}.mark i:nth-child(3){opacity:.5}.badge{border:1px solid var(--line);border-radius:20px;padding:7px 12px;font-size:11px;letter-spacing:.8px;color:var(--muted)}.status{color:var(--teal);display:flex;align-items:center;gap:8px}.dot{width:6px;height:6px;background:currentColor;border-radius:50%}.layout{display:grid;grid-template-columns:268px minmax(0,1fr);min-height:calc(100vh - 86px)}aside{padding:23px 20px;border-right:1px solid var(--line)}.eyebrow{font-size:10px;font-weight:700;letter-spacing:1.6px;color:var(--muted);text-transform:uppercase;margin:0 0 18px}label{display:block;font-size:12px;color:var(--muted);margin:18px 0 8px}select,input[type=number]{font:inherit;color:var(--text);width:100%;background:#142438;border:1px solid #30465e;padding:11px;border-radius:7px;outline:none}select:focus,input:focus{border-color:var(--teal)}input[type=range]{width:100%;accent-color:var(--teal);cursor:pointer}.sliderlabels{display:flex;justify-content:space-between;font-size:10px;color:var(--muted)}.row{display:flex;align-items:center;justify-content:space-between;gap:8px}.value{color:var(--teal);font-weight:650}button{font:inherit;border:1px solid #365069;background:#182b40;color:var(--text);padding:10px 14px;border-radius:7px;cursor:pointer;transition:background .15s}button:hover{background:#24415d}button:disabled{opacity:.4;cursor:default}.primary{width:100%;background:var(--teal);border-color:var(--teal);color:#06231f;font-weight:700;margin-top:22px}.primary:hover{background:#8be9d5}.note{color:var(--muted);font-size:11px;line-height:1.65}.divider{height:1px;background:var(--line);margin:24px 0}.inspector{font-size:12px;line-height:1.9}.approach{display:flex;justify-content:space-between;border-bottom:1px solid #1b2d40;padding:4px 0}.green{color:var(--teal)}main{padding:22px 26px;min-width:0}.topline{display:flex;justify-content:space-between;align-items:center;margin-bottom:18px}.topline h2{font-size:15px;margin:0}.clock{font-variant-numeric:tabular-nums;font-size:13px;color:var(--muted)}.kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:16px}.kpi{background:var(--panel);border:1px solid var(--line);padding:15px 17px;border-radius:9px}.kpi .number{font-size:28px;letter-spacing:-1px;font-variant-numeric:tabular-nums;margin:4px 0}.kpi .caption{font-size:11px;color:var(--muted)}.kpi .detail{font-size:10px;color:var(--muted)}.mapcard{border:1px solid var(--line);background:#0c1827;border-radius:10px;overflow:hidden}.mapbar{padding:12px 16px;border-bottom:1px solid var(--line);display:flex;justify-content:space-between;align-items:center;gap:10px}.mapbar .title{font-size:12px}.legend{display:flex;align-items:center;gap:12px;font-size:10px;color:var(--muted)}.legend b{display:inline-block;width:6px;height:6px;border-radius:2px;margin-right:4px}.mapwrap{height:clamp(360px,49vh,650px);position:relative}#map{width:100%;height:100%;display:block;cursor:crosshair}.maphint{position:absolute;left:15px;bottom:12px;font-size:10px;color:#7790a9;pointer-events:none}.controls{display:flex;align-items:center;gap:7px;padding:12px 15px;border-top:1px solid var(--line);flex-wrap:wrap}.controls button{padding:8px 13px;font-size:12px}.speed{display:flex;align-items:center;gap:9px;margin-left:auto;color:var(--muted);font-size:11px}.speed select{width:76px;padding:6px;font-size:12px}.chartcard{margin-top:14px;padding:15px 16px;background:var(--panel);border:1px solid var(--line);border-radius:9px}.charthead{display:flex;align-items:center;justify-content:space-between;font-size:12px}.chartwrap{height:105px;margin-top:6px}#chart{display:block;width:100%;height:100%}.footer{display:flex;justify-content:space-between;font-size:10px;color:var(--muted);padding-top:12px;gap:15px}.error{display:none;background:#3b1c28;color:#ffc0c0;border:1px solid #9e4456;padding:12px;margin-bottom:14px;border-radius:7px;font-size:12px}.disclosure{font-size:10px;letter-spacing:.6px;color:var(--muted)}@media(max-width:950px){header{padding:17px}.layout{grid-template-columns:220px minmax(0,1fr)}aside{padding:20px 14px}main{padding:18px 14px}.kpis{gap:7px}.kpi{padding:12px}.kpi .number{font-size:23px}.legend{gap:7px}.badge{display:none}}@media(max-width:650px){.layout{display:flex;flex-direction:column}aside{border-right:0;border-bottom:1px solid var(--line);display:grid;grid-template-columns:1fr 1fr;gap:0 14px}aside>.eyebrow,aside>.primary,aside>.note,aside>.divider,aside>.inspector{grid-column:1/-1}aside>.divider,aside>.inspector{display:none}label{margin-top:12px}.kpis{grid-template-columns:repeat(2,1fr)}.mapwrap{height:420px}.mapbar{flex-wrap:wrap}.disclosure{display:none}}\n</style></head><body>\n<header><div class="brand"><div class="mark" aria-hidden="true"><i></i><i></i><i></i><i></i></div><div><h1>Adaptive Signals</h1><p>Traffic control laboratory · live SUMO simulation</p></div></div><span class="badge">SYNTHETIC NETWORK</span><div class="status" id="status"><i class="dot"></i><span>Connecting</span></div></header>\n<div class="layout"><aside>\n<p class="eyebrow">Experiment setup</p>\n<div><label for="controller">Signal controller</label><select id="controller"><option value="predictive">Discharge lookahead</option><option value="coordinated">Stage-1 coordinated</option><option value="calibrated_pressure">Turn-calibrated pressure</option><option value="sumo_actuated">SUMO actuated reference</option><option value="fixed">Development-tuned fixed</option><option value="receiving_aware">Receiving capacity only</option><option value="message_only">Upstream messages only</option><option value="moving_only">Moving vehicles only</option><option value="max_pressure">Equal-turn pressure</option></select></div>\n<div><label for="scenario">Traffic pattern</label><select id="scenario"><option value="directional_switch">Changing direction</option><option value="balanced">Balanced demand</option><option value="surge">Demand surge</option></select></div>\n<div><label class="row" for="load">Demand multiplier <span class="value" id="loadValue">0.6×</span></label><input type="range" id="load" min="0.1" max="1.5" value="0.6" step="0.1"><div class="sliderlabels"><span>Lighter</span><span>Heavier</span></div></div>\n<div><label for="regime">Controller sensors</label><select id="regime"><option value="perfect">Exact lane counts</option><option value="degraded">Delayed, noisy & missing</option></select></div>\n<div><label for="duration">Traffic arrival period</label><select id="duration"><option value="300">5 minutes</option><option value="600" selected>10 minutes</option><option value="900">15 minutes</option><option value="1800">30 minutes</option></select></div>\n<div><label for="seed">Traffic seed</label><input id="seed" type="number" min="0" max="2147483647" value="12777"></div>\n<button class="primary" id="start">Start / reset run</button><p class="note">Settings apply on reset. Reuse the seed to compare identical traffic requests. Sensor settings affect the controller; counters show simulation truth.</p>\n<div class="divider"></div><div class="inspector"><p class="eyebrow" id="nodeTitle">Intersection B1</p><div id="approaches">Select an intersection on the map.</div></div>\n</aside><main>\n<div class="topline"><h2>Network overview</h2><span class="clock" id="clock">00:00 simulation time</span></div>\n<div class="error" id="error" role="alert"></div>\n<div class="kpis"><div class="kpi"><div class="caption">Completed trips</div><div class="number" id="arrived">—</div><div class="detail" id="requested">Awaiting a live run</div></div><div class="kpi"><div class="caption">Vehicles in network</div><div class="number" id="active">—</div><div class="detail" id="pending">Includes internal junctions</div></div><div class="kpi"><div class="caption">Stopped vehicles</div><div class="number" id="halted">—</div><div class="detail">Speed below 0.1 m/s on roads</div></div><div class="kpi"><div class="caption">Mean system time so far</div><div class="number" id="system">—</div><div class="detail">All requests made so far</div></div></div>\n<section class="mapcard"><div class="mapbar"><span class="title">Nine connected intersections</span><div class="legend"><span><b style="background:#54d5bd"></b>Moving</span><span><b style="background:#f7ba58"></b>Stopped</span><span><b style="background:#f17b7b"></b>Red signal</span></div><label style="margin:0;font-size:10px"><input type="checkbox" id="overlay" checked> Detector queues</label></div><div class="mapwrap"><canvas id="map" aria-label="Live traffic simulation map"></canvas><span class="maphint">Click an intersection to inspect its approach queues.</span></div><div class="controls"><button id="pause" disabled>Pause</button><button id="resume" disabled>Resume</button><button id="step" disabled>Step 1 second</button><button id="stop" disabled>Stop</button><div class="speed"><label for="speed" style="margin:0">Playback rate</label><select id="speed"><option value="1">1×</option><option value="5">5×</option><option value="10" selected>10×</option><option value="30">30×</option><option value="60">60×</option></select></div></div></section>\n<section class="chartcard"><div class="charthead"><span>Queue and entry backlog</span><div class="legend"><span><b style="background:#f7ba58"></b>Stopped on roads</span><span><b style="background:#869eec"></b>Waiting to enter</span></div></div><div class="chartwrap"><canvas id="chart" aria-label="Queue and entry backlog over time"></canvas></div></section>\n<div class="footer"><span id="message">Actual vehicle positions from a running SUMO process.</span><span id="audit">Collision / teleport monitoring active</span></div>\n</main></div>\n<script>\n\'use strict\';\nconst ACCESS_KEY=__ACCESS_KEY__;\nconst $=id=>document.getElementById(id);\nlet scene=null,state=null,selected=\'B1\',busy=false,connected=false;\nconst base=new URL(\'./\',location.href);\nasync function api(path,body){const response=await fetch(new URL(\'api/\'+path,base),{method:body?\'POST\':\'GET\',headers:{\'X-Simulation-Key\':ACCESS_KEY,...(body?{\'Content-Type\':\'application/json\'}:{})},...(body?{body:JSON.stringify(body)}:{})});const data=await response.json();if(!response.ok)throw new Error(data.error||\'The simulation request failed.\');return data;}\nfunction error(message){$(\'error\').textContent=message;$(\'error\').style.display=message?\'block\':\'none\';}\nasync function command(action,values){try{error(\'\');await api(\'control\',{action,values});}catch(e){error(e.message);}}\n$(\'load\').oninput=()=>{$(\'loadValue\').textContent=Number($(\'load\').value).toFixed(1)+\'×\';};\n$(\'start\').onclick=()=>command(\'start\',{controller:$(\'controller\').value,scenario:$(\'scenario\').value,scale:Number($(\'load\').value),regime:$(\'regime\').value,seed:Number($(\'seed\').value),demand_s:Number($(\'duration\').value)});\nfor(const id of [\'pause\',\'resume\',\'step\',\'stop\'])$(id).onclick=()=>command(id);\n$(\'speed\').onchange=()=>command(\'speed\',{speed:Number($(\'speed\').value)});\n$(\'overlay\').onchange=draw;\nfunction size(canvas){const r=canvas.getBoundingClientRect(),d=window.devicePixelRatio||1;canvas.width=Math.round(r.width*d);canvas.height=Math.round(r.height*d);const ctx=canvas.getContext(\'2d\');ctx.setTransform(d,0,0,d,0,0);return {ctx,w:r.width,h:r.height};}\nlet project=null;\nfunction draw(){const {ctx,w,h}=size($(\'map\'));ctx.fillStyle=\'#0c1827\';ctx.fillRect(0,0,w,h);if(!scene)return;const [xmin,ymin,xmax,ymax]=scene.bounds;const scale=Math.min((w-64)/(xmax-xmin),(h-46)/(ymax-ymin));const ox=(w-(xmax-xmin)*scale)/2,oy=(h-(ymax-ymin)*scale)/2;project=(x,y)=>[ox+(x-xmin)*scale,h-oy-(y-ymin)*scale];\n function line(points,color,width){ctx.beginPath();points.forEach((p,i)=>{const [x,y]=project(...p);if(i)ctx.lineTo(x,y);else ctx.moveTo(x,y);});ctx.strokeStyle=color;ctx.lineWidth=width;ctx.stroke();}\n ctx.lineCap=\'butt\';for(const lane of scene.lanes){line(lane.shape,\'#24384e\',Math.max(3,4.2*scale));}\n const signalMap=new Map(state?.signals||[]);\n for(const lane of scene.lanes){const points=lane.shape,last=points[points.length-1],prev=points[points.length-2];if(!prev)continue;const end=project(...last),p=project(...prev);const dx=end[0]-p[0],dy=end[1]-p[1],len=Math.hypot(dx,dy)||1,ux=dx/len,uy=dy/len;const observed=state?.sensors?.[lane.id]?.[0]||0;\n  if($(\'overlay\').checked&&observed>0){const length=Math.min(lane.length-10,observed*7.5)*scale;ctx.beginPath();ctx.moveTo(...end);ctx.lineTo(end[0]-ux*length,end[1]-uy*length);ctx.lineWidth=Math.max(4,5*scale);ctx.strokeStyle=\'rgba(247,186,88,.32)\';ctx.stroke();}\n  const s=signalMap.get(lane.id);if(s){ctx.beginPath();ctx.moveTo(end[0]-uy*4,end[1]+ux*4);ctx.lineTo(end[0]+uy*4,end[1]-ux*4);ctx.lineWidth=2.5;ctx.strokeStyle=s===\'G\'?\'#54d5bd\':s===\'y\'?\'#f7ba58\':\'#f17b7b\';ctx.stroke();}}\n for(const node of scene.nodes){const [x,y]=project(...node.xy);if(node.id===selected){ctx.strokeStyle=\'rgba(84,213,189,.6)\';ctx.lineWidth=1;ctx.beginPath();ctx.arc(x,y,17,0,Math.PI*2);ctx.stroke();}ctx.fillStyle=\'#a6bacf\';ctx.font=\'10px system-ui\';ctx.fillText(node.id,x+13,y-12);}\n for(const v of state?.vehicles||[]){const [id,x,y,angle,speed]=v,[px,py]=project(x,y);ctx.save();ctx.translate(px,py);ctx.rotate(angle*Math.PI/180-Math.PI/2);ctx.fillStyle=speed<.1?\'#f7ba58\':speed<5?\'#b6d784\':\'#54d5bd\';ctx.fillRect(-Math.max(4.5,5*scale)/2,-Math.max(2,2*scale)/2,Math.max(4.5,5*scale),Math.max(2,2*scale));ctx.restore();}\n drawChart();inspect();}\nfunction inspect(){if(!scene)return;$(\'nodeTitle\').textContent=\'Intersection \'+selected;const incoming=scene.lanes.filter(l=>l.to===selected),signals=new Map(state?.signals||[]);$(\'approaches\').replaceChildren();for(const lane of incoming){const div=document.createElement(\'div\');div.className=\'approach\';const a=document.createElement(\'span\'),b=document.createElement(\'span\');a.textContent=lane.from+\' → \'+selected;b.textContent=state?.sensors?((state.sensors[lane.id]?.[0]||0).toFixed(1)+\' vehicles\'):\'—\';if(signals.get(lane.id)===\'G\'){a.className=\'green\';b.className=\'green\';}div.append(a,b);$(\'approaches\').append(div);}}\nfunction drawChart(){const {ctx,w,h}=size($(\'chart\'));ctx.clearRect(0,0,w,h);const rows=state?.history||[];const left=30,top=8,bottom=h-18,maxT=Math.max(60,state?.time_s||0),maxY=Math.max(10,...rows.flatMap(r=>[r[1],r[2]]));ctx.font=\'9px system-ui\';ctx.strokeStyle=\'#26394c\';ctx.fillStyle=\'#8da4bb\';for(let i=0;i<3;i++){const val=maxY*i/2,y=bottom-(bottom-top)*i/2;ctx.beginPath();ctx.moveTo(left,y);ctx.lineTo(w-5,y);ctx.stroke();ctx.fillText(String(Math.round(val)),0,y+3);}for(const [idx,color]of [[1,\'#f7ba58\'],[2,\'#869eec\']]){ctx.beginPath();rows.forEach((r,i)=>{const x=left+r[0]/maxT*(w-left-5),y=bottom-r[idx]/maxY*(bottom-top);if(i)ctx.lineTo(x,y);else ctx.moveTo(x,y);});ctx.strokeStyle=color;ctx.lineWidth=1.8;ctx.stroke();}ctx.fillText(\'0 s\',left,h-3);ctx.fillText(Math.round(maxT)+\' s\',w-40,h-3);}\n$(\'map\').onclick=event=>{if(!scene||!project)return;const r=$(\'map\').getBoundingClientRect(),x=event.clientX-r.left,y=event.clientY-r.top;let best=null,d=30;for(const n of scene.nodes){const p=project(...n.xy),distance=Math.hypot(x-p[0],y-p[1]);if(distance<d){best=n;d=distance;}}if(best){selected=best.id;draw();}};\nfunction render(){if(!state)return;const m=state.metrics||{},started=state.time_s>0;const status=state.status;$(\'status\').lastElementChild.textContent=status[0].toUpperCase()+status.slice(1);$(\'status\').style.color=status===\'error\'?\'#f17b7b\':status===\'paused\'?\'#f7ba58\':\'#54d5bd\';const t=state.time_s||0;$(\'clock\').textContent=String(Math.floor(t/60)).padStart(2,\'0\')+\':\'+String(t%60).padStart(2,\'0\')+\' simulation time\';for(const id of [\'arrived\',\'active\',\'halted\'])$(id).textContent=started?(m[id]??0):\'—\';$(\'system\').textContent=started?(m.mean_system_so_far_s??0)+\' s\':\'—\';$(\'requested\').textContent=started?`${m.requested} requested so far / ${m.total_requests} total`:\'Awaiting a live run\';$(\'pending\').textContent=started?`${m.pending} waiting to enter`:\'Includes internal junctions\';$(\'message\').textContent=state.message||\'Actual SUMO vehicle positions · detector overlay uses controller observations.\';$(\'audit\').textContent=started?`${m.collisions||0} collision events · ${m.teleports||0} teleports`:\'Collision / teleport monitoring active\';$(\'pause\').disabled=status!==\'running\';$(\'resume\').disabled=status!==\'paused\';$(\'step\').disabled=![\'running\',\'paused\'].includes(status);$(\'stop\').disabled=![\'running\',\'paused\',\'starting\'].includes(status);if(status===\'error\')error(state.message);draw();}\nasync function poll(){if(busy)return;busy=true;try{if(!scene)scene=await api(\'scene\');state=await api(\'state\');connected=true;render();}catch(e){$(\'status\').lastElementChild.textContent=\'Disconnected\';$(\'status\').style.color=\'#f17b7b\';error(\'Live connection unavailable. Keep the Python/Colab simulation session running. \'+e.message);}finally{busy=false;}}\nwindow.addEventListener(\'resize\',draw);poll();setInterval(poll,200);\n</script></body></html>\n', 'tests/test_invariants.py': '"""Checks for consequential evaluation/controller failures, not implementation coverage."""\nimport sys\nfrom pathlib import Path\nimport tempfile\nimport unittest\nimport csv\nimport json\n\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nimport experiment as e\n\n\nclass KernelTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls): cls.k=e.Kernel()\n\n    def test_green_bounds_override_scores(self):\n        self.assertFalse(self.k.switch(-1000,1000,9))\n        self.assertTrue(self.k.switch(1000,-1000,50))\n\n    def test_downstream_queue_changes_pressure(self):\n        self.assertGreater(self.k.score(1,[(10,1,0,0,0)]),self.k.score(1,[(10,8,0,0,0)]))\n\n    def test_full_receiving_lane_removes_positive_release_incentive(self):\n        self.assertEqual(self.k.score(2,[(20,0,1,0,0)]),0)\n        self.assertGreater(self.k.score(2,[(20,0,0,0,0)]),0)\n\n    def test_announcements_only_affect_coordinated_controller(self):\n        base=[(3,1,0.2,0,0)];msg=[(3,1,0.2,5,0)]\n        self.assertEqual(self.k.score(2,base),self.k.score(2,msg))\n        self.assertGreater(self.k.score(3,msg),self.k.score(3,base))\n\n\nclass IntegrationTests(unittest.TestCase):\n    def test_demand_identity_determinism_and_signal_clearance(self):\n        with tempfile.TemporaryDirectory() as d:\n            root=Path(d)\n            netfile,net,tls=e.build_network(root/\'inputs\')\n            demand,schedule=e.generate_demand(root/\'inputs\',net,\'surge\',777,120)\n            before=e.sha(demand)\n            demand2,schedule2=e.generate_demand(root/\'inputs\',net,\'surge\',777,120)\n            self.assertEqual(before,e.sha(demand2));self.assertEqual(schedule,schedule2)\n            results=[]\n            for i in range(2):\n                results.append(e.run_one(root/f\'run{i}\',netfile,tls,demand,schedule,\n                    \'coordinated\',\'degraded\',777,120,120,30,False))\n            for key in [\'total_system_time_s\',\'arrived\',\'active\',\'pending\',\'switches\']:\n                self.assertEqual(results[0][key],results[1][key])\n            self.assertTrue(results[0][\'valid_run\'])\n            self.assertTrue(results[0][\'metric_ledger_pass\'])\n            self.assertEqual(results[0][\'scheduled\'],results[0][\'arrived\']+results[0][\'active\']+results[0][\'pending\'])\n            with (root/\'run0/signals.csv\').open() as f:\n                for row in csv.DictReader(f):\n                    old,new=int(row[\'old_phase\']),int(row[\'new_phase\'])\n                    if old%3==2:self.assertEqual(new%3,0)\n                    else:self.assertEqual(new,old+1)\n                    duration=float(row[\'old_duration_s\'])\n                    if old%3==0:self.assertGreaterEqual(duration,10);self.assertLessEqual(duration,50)\n                    elif old%3==1:self.assertEqual(duration,3)\n                    else:self.assertEqual(duration,2)\n\n    def test_uninserted_and_unfinished_requests_cannot_disappear_from_metric(self):\n        with tempfile.TemporaryDirectory() as d:\n            root=Path(d)\n            netfile,net,tls=e.build_network(root/\'inputs\')\n            demand,schedule=e.generate_demand(root/\'inputs\',net,\'balanced\',1777,120)\n            # Saturate one source at t=0 so many requests cannot enter by the horizon.\n            import xml.etree.ElementTree as ET\n            xml=ET.parse(demand);r=xml.getroot()\n            template=r.find(\'vehicle\')\n            for i in range(400):\n                v=ET.fromstring(ET.tostring(template));v.set(\'id\',f\'stress{i}\');v.set(\'depart\',\'0\')\n                r.append(v)\n            vehicles=sorted(r.findall(\'vehicle\'),key=lambda v:float(v.attrib[\'depart\']))\n            for v in r.findall(\'vehicle\'):r.remove(v)\n            r.extend(vehicles);xml.write(demand)\n            schedule={v.attrib[\'id\']:float(v.attrib[\'depart\']) for v in vehicles}\n            result=e.run_one(root/\'stress\',netfile,tls,demand,schedule,\n                \'fixed\',\'perfect\',1777,120,0,30,False)\n            self.assertGreater(result[\'pending\'],100)\n            self.assertGreaterEqual(result[\'total_system_time_s\'],result[\'pending\']*120)\n            self.assertTrue(result[\'metric_ledger_pass\'])\n\n\nif __name__==\'__main__\':unittest.main(verbosity=2)\n', 'tests/test_stage2.py': 'import json\nfrom pathlib import Path\nimport re\nimport sys\nimport tempfile\nimport threading\nimport time\nimport unittest\nimport urllib.error\nimport urllib.request\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nimport experiment as e\nfrom live_server import LiveApp,server\nfrom stage2 import learn_turns\n\n\nclass ExtendedControlTests(unittest.TestCase):\n    def test_calibration_uses_only_supplied_development_routes(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root=Path(tmp);nf,net,tls=e.build_network(root/\'in\')\n            d,s=e.generate_demand(root/\'in\',net,\'balanced\',201,120,0.3)\n            turns=learn_turns([d]);self.assertTrue(turns)\n            for outcomes in turns.values():self.assertAlmostEqual(sum(outcomes.values()),1)\n            obs={\'in\':(10,12),\'a\':(1,2),\'b\':(9,10)}\n            rows=e.phase_rows({\'in\':[\'a\',\'b\']},obs,{\'in\':20,\'a\':20,\'b\':20},{\'in\':0},{\'in\':{\'a\':1,\'b\':0}})\n            self.assertEqual(rows[0][1],1)\n\n    def test_lost_time_reduces_discharge_benefit(self):\n        k=e.Kernel({\'horizon_s\':20,\'receiving_power\':1,\'message_weight\':0.3,\'moving_weight\':0.2})\n        self.assertGreater(k.predictive([(8,0,0,0,0)],0),k.predictive([(8,0,0,0,0)],5))\n\n    def test_native_actuated_and_predictive_preserve_conservation_and_clearance(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root=Path(tmp);nf,net,tls=e.build_network(root/\'in\')\n            d,s=e.generate_demand(root/\'in\',net,\'balanced\',201,120,0.3)\n            for c in [\'sumo_actuated\',\'predictive\']:\n                r=e.run_one(root/c,nf,tls,d,s,c,\'perfect\',201,120,900,10,False,stop_when_clear=True)\n                self.assertTrue(r[\'valid_run\']);self.assertTrue(r[\'metric_ledger_pass\'])\n                self.assertTrue(r[\'cleared\'])\n                if c==\'sumo_actuated\':\n                    import csv\n                    with (root/c/\'signals.csv\').open() as f:\n                        durations=[float(x[\'old_duration_s\']) for x in csv.DictReader(f) if int(x[\'old_phase\'])%3==0]\n                    self.assertTrue(any(x<20 for x in durations),\'Native reference retained static 30-second greens\')\n\n\nclass LiveHTTPTests(unittest.TestCase):\n    def test_real_simulation_pause_step_reset_stop_and_input_validation(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            app=LiveApp(tmp);http=server(app,port=0)\n            thread=threading.Thread(target=http.serve_forever,daemon=True);thread.start()\n            base=f\'http://127.0.0.1:{http.server_port}/\'\n            try:\n                html=urllib.request.urlopen(base).read().decode()\n                key=json.loads(re.search(r\'const ACCESS_KEY=("[^"]+");\',html).group(1))\n                def call(path,body=None,key_=key):\n                    req=urllib.request.Request(base+\'api/\'+path,\n                        data=json.dumps(body).encode() if body else None,\n                        headers={\'X-Simulation-Key\':key_,\'Content-Type\':\'application/json\'})\n                    with urllib.request.urlopen(req,timeout=5) as r:return json.load(r)\n                def control(action,values=None):return call(\'control\',{\'action\':action,\'values\':values})\n                def until(predicate,timeout=10):\n                    deadline=time.monotonic()+timeout\n                    while time.monotonic()<deadline:\n                        value=call(\'state\')\n                        if predicate(value):return value\n                        if value[\'status\']==\'error\':self.fail(value[\'message\'])\n                        time.sleep(.03)\n                    self.fail(\'Live state did not reach expected condition\')\n                with self.assertRaises(urllib.error.HTTPError):call(\'state\',key_=\'wrong\')\n                with self.assertRaises(urllib.error.HTTPError):control(\'start\',{\'scale\':99})\n                control(\'speed\',{\'speed\':60})\n                control(\'start\',{\'controller\':\'predictive\',\'demand_s\':60,\'scale\':0.3,\'seed\':12777})\n                state=until(lambda s:s[\'time_s\']>=8)\n                self.assertGreater(state[\'metrics\'][\'requested\'],0)\n                self.assertIsInstance(state[\'vehicles\'],list)\n                control(\'pause\');time.sleep(.15);paused=call(\'state\');t=paused[\'time_s\']\n                time.sleep(.15);self.assertEqual(call(\'state\')[\'time_s\'],t)\n                self.assertEqual(call(\'state\')[\'status\'],\'paused\')\n                control(\'step\');stepped=until(lambda s:s[\'time_s\']==t+1)\n                time.sleep(.1);self.assertEqual(call(\'state\')[\'time_s\'],t+1)\n                control(\'start\',{\'seed\':13777,\'demand_s\':60,\'scale\':0.3})\n                reset=until(lambda s:s[\'config\'][\'seed\']==13777 and s[\'time_s\']>=2 and s[\'time_s\']<8)\n                control(\'stop\');until(lambda s:s[\'status\']==\'stopped\')\n                self.assertEqual(reset[\'metrics\'][\'collisions\'],0)\n            finally:\n                http.shutdown();http.server_close();app.close();thread.join(timeout=2)\n\n\nif __name__==\'__main__\':unittest.main(verbosity=2)\n', 'tests/check_viewer.cjs': '// Numerical/DOM smoke check with supplied SUMO-shaped data, not a browser test.\nconst fs=require(\'node:fs\'),vm=require(\'node:vm\');\nconst [htmlPath,fixturePath]=process.argv.slice(2);\nconst fixture=JSON.parse(fs.readFileSync(fixturePath,\'utf8\'));\nconst source=fs.readFileSync(htmlPath,\'utf8\').split(\'<script>\')[1].split(\'</script>\')[0].replace(\'__ACCESS_KEY__\',\'"test-key"\');\nlet drawCalls=0;\nconst numeric=(...args)=>{if(args.some(v=>typeof v!==\'number\'||!Number.isFinite(v)))throw Error(\'Nonfinite canvas coordinate\');drawCalls++;};\nconst ctx={setTransform:numeric,fillRect:numeric,clearRect:numeric,moveTo:numeric,lineTo:numeric,\n arc:numeric,translate:numeric,rotate:numeric,beginPath(){},stroke(){},save(){},restore(){},\n fillText(text,x,y){numeric(x,y);}};\nconst elements=new Map();\nfunction element(id){if(!elements.has(id))elements.set(id,{id,value:id===\'load\'?\'0.6\':\'10\',checked:true,\n style:{},lastElementChild:{textContent:\'\'},append(){},replaceChildren(){},getContext(){return ctx;},\n getBoundingClientRect(){return {width:900,height:id===\'chart\'?105:600,left:0,top:0};}});return elements.get(id);}\nconst sandbox={console,URL,Math,Map,Number,String,JSON,Error,\n document:{getElementById:element,createElement(){return {append(){}};}},\n window:{devicePixelRatio:2,addEventListener(){}},location:{href:\'http://local.test/\'},\n fetch:async url=>({ok:true,json:async()=>String(url).endsWith(\'scene\')?fixture.scene:fixture.state}),\n setInterval(){},setTimeout};\nvm.createContext(sandbox);vm.runInContext(source,sandbox);\nsetTimeout(()=>{\n try{\n  vm.runInContext(\'draw(); inspect(); render();\',sandbox);\n  if(drawCalls<100)throw Error(\'Rendering did not exercise scene\');\n  if(element(\'arrived\').textContent!==fixture.state.metrics.arrived)throw Error(\'Metrics binding failed\');\n  console.log(\'VIEWER_NUMERICAL_SMOKE_PASS\',drawCalls,\'canvas calls\');\n }catch(e){console.error(e);process.exitCode=1;}\n},30);\n', 'tests/viewer_fixture.json': '{"scene": {"bounds": [0.0, -0.0, 760.0, 760.0], "lanes": [{"id": "A0A1_0", "shape": [[218.4, 227.2], [218.4, 372.8]], "length": 145.6, "from": "A0", "to": "A1"}, {"id": "A0B0_0", "shape": [[227.2, 221.6], [372.8, 221.6]], "length": 145.6, "from": "A0", "to": "B0"}, {"id": "A0bottom0_0", "shape": [[221.6, 212.8], [221.6, -0.0]], "length": 212.8, "from": "A0", "to": "bottom0"}, {"id": "A0left0_0", "shape": [[212.8, 218.4], [0.0, 218.4]], "length": 212.8, "from": "A0", "to": "left0"}, {"id": "A1A0_0", "shape": [[221.6, 372.8], [221.6, 227.2]], "length": 145.6, "from": "A1", "to": "A0"}, {"id": "A1A2_0", "shape": [[218.4, 387.2], [218.4, 532.8]], "length": 145.6, "from": "A1", "to": "A2"}, {"id": "A1B1_0", "shape": [[227.2, 381.6], [372.8, 381.6]], "length": 145.6, "from": "A1", "to": "B1"}, {"id": "A1left1_0", "shape": [[212.8, 378.4], [0.0, 378.4]], "length": 212.8, "from": "A1", "to": "left1"}, {"id": "A2A1_0", "shape": [[221.6, 532.8], [221.6, 387.2]], "length": 145.6, "from": "A2", "to": "A1"}, {"id": "A2B2_0", "shape": [[227.2, 541.6], [372.8, 541.6]], "length": 145.6, "from": "A2", "to": "B2"}, {"id": "A2left2_0", "shape": [[212.8, 538.4], [0.0, 538.4]], "length": 212.8, "from": "A2", "to": "left2"}, {"id": "A2top0_0", "shape": [[218.4, 547.2], [218.4, 760.0]], "length": 212.8, "from": "A2", "to": "top0"}, {"id": "B0A0_0", "shape": [[372.8, 218.4], [227.2, 218.4]], "length": 145.6, "from": "B0", "to": "A0"}, {"id": "B0B1_0", "shape": [[378.4, 227.2], [378.4, 372.8]], "length": 145.6, "from": "B0", "to": "B1"}, {"id": "B0C0_0", "shape": [[387.2, 221.6], [532.8, 221.6]], "length": 145.6, "from": "B0", "to": "C0"}, {"id": "B0bottom1_0", "shape": [[381.6, 212.8], [381.6, -0.0]], "length": 212.8, "from": "B0", "to": "bottom1"}, {"id": "B1A1_0", "shape": [[372.8, 378.4], [227.2, 378.4]], "length": 145.6, "from": "B1", "to": "A1"}, {"id": "B1B0_0", "shape": [[381.6, 372.8], [381.6, 227.2]], "length": 145.6, "from": "B1", "to": "B0"}, {"id": "B1B2_0", "shape": [[378.4, 387.2], [378.4, 532.8]], "length": 145.6, "from": "B1", "to": "B2"}, {"id": "B1C1_0", "shape": [[387.2, 381.6], [532.8, 381.6]], "length": 145.6, "from": "B1", "to": "C1"}, {"id": "B2A2_0", "shape": [[372.8, 538.4], [227.2, 538.4]], "length": 145.6, "from": "B2", "to": "A2"}, {"id": "B2B1_0", "shape": [[381.6, 532.8], [381.6, 387.2]], "length": 145.6, "from": "B2", "to": "B1"}, {"id": "B2C2_0", "shape": [[387.2, 541.6], [532.8, 541.6]], "length": 145.6, "from": "B2", "to": "C2"}, {"id": "B2top1_0", "shape": [[378.4, 547.2], [378.4, 760.0]], "length": 212.8, "from": "B2", "to": "top1"}, {"id": "C0B0_0", "shape": [[532.8, 218.4], [387.2, 218.4]], "length": 145.6, "from": "C0", "to": "B0"}, {"id": "C0C1_0", "shape": [[538.4, 227.2], [538.4, 372.8]], "length": 145.6, "from": "C0", "to": "C1"}, {"id": "C0bottom2_0", "shape": [[541.6, 212.8], [541.6, -0.0]], "length": 212.8, "from": "C0", "to": "bottom2"}, {"id": "C0right0_0", "shape": [[547.2, 221.6], [760.0, 221.6]], "length": 212.8, "from": "C0", "to": "right0"}, {"id": "C1B1_0", "shape": [[532.8, 378.4], [387.2, 378.4]], "length": 145.6, "from": "C1", "to": "B1"}, {"id": "C1C0_0", "shape": [[541.6, 372.8], [541.6, 227.2]], "length": 145.6, "from": "C1", "to": "C0"}, {"id": "C1C2_0", "shape": [[538.4, 387.2], [538.4, 532.8]], "length": 145.6, "from": "C1", "to": "C2"}, {"id": "C1right1_0", "shape": [[547.2, 381.6], [760.0, 381.6]], "length": 212.8, "from": "C1", "to": "right1"}, {"id": "C2B2_0", "shape": [[532.8, 538.4], [387.2, 538.4]], "length": 145.6, "from": "C2", "to": "B2"}, {"id": "C2C1_0", "shape": [[541.6, 532.8], [541.6, 387.2]], "length": 145.6, "from": "C2", "to": "C1"}, {"id": "C2right2_0", "shape": [[547.2, 541.6], [760.0, 541.6]], "length": 212.8, "from": "C2", "to": "right2"}, {"id": "C2top2_0", "shape": [[538.4, 547.2], [538.4, 760.0]], "length": 212.8, "from": "C2", "to": "top2"}, {"id": "bottom0A0_0", "shape": [[218.4, -0.0], [218.4, 212.8]], "length": 212.8, "from": "bottom0", "to": "A0"}, {"id": "bottom1B0_0", "shape": [[378.4, -0.0], [378.4, 212.8]], "length": 212.8, "from": "bottom1", "to": "B0"}, {"id": "bottom2C0_0", "shape": [[538.4, -0.0], [538.4, 212.8]], "length": 212.8, "from": "bottom2", "to": "C0"}, {"id": "left0A0_0", "shape": [[0.0, 221.6], [212.8, 221.6]], "length": 212.8, "from": "left0", "to": "A0"}, {"id": "left1A1_0", "shape": [[0.0, 381.6], [212.8, 381.6]], "length": 212.8, "from": "left1", "to": "A1"}, {"id": "left2A2_0", "shape": [[0.0, 541.6], [212.8, 541.6]], "length": 212.8, "from": "left2", "to": "A2"}, {"id": "right0C0_0", "shape": [[760.0, 218.4], [547.2, 218.4]], "length": 212.8, "from": "right0", "to": "C0"}, {"id": "right1C1_0", "shape": [[760.0, 378.4], [547.2, 378.4]], "length": 212.8, "from": "right1", "to": "C1"}, {"id": "right2C2_0", "shape": [[760.0, 538.4], [547.2, 538.4]], "length": 212.8, "from": "right2", "to": "C2"}, {"id": "top0A2_0", "shape": [[221.6, 760.0], [221.6, 547.2]], "length": 212.8, "from": "top0", "to": "A2"}, {"id": "top1B2_0", "shape": [[381.6, 760.0], [381.6, 547.2]], "length": 212.8, "from": "top1", "to": "B2"}, {"id": "top2C2_0", "shape": [[541.6, 760.0], [541.6, 547.2]], "length": 212.8, "from": "top2", "to": "C2"}], "nodes": [{"id": "A0", "xy": [220.0, 220.0]}, {"id": "A1", "xy": [220.0, 380.0]}, {"id": "A2", "xy": [220.0, 540.0]}, {"id": "B0", "xy": [380.0, 220.0]}, {"id": "B1", "xy": [380.0, 380.0]}, {"id": "B2", "xy": [380.0, 540.0]}, {"id": "C0", "xy": [540.0, 220.0]}, {"id": "C1", "xy": [540.0, 380.0]}, {"id": "C2", "xy": [540.0, 540.0]}]}, "state": {"status": "running", "time_s": 230, "vehicles": [["v000006", 63.76545999999999, 538.4, 270.0, 0], ["v000011", 95.55198320000002, 538.4, 270.0, 0], ["v000017", 218.4, 531.799, 0.0, 0], ["v000018", 218.4, 731.5374338873053, 0.0, 0], ["v000021", 221.6, 425.50815265527143, 180.0, 0], ["v000036", 243.203, 218.4, 270.0, 0], ["v000042", 250.704, 218.4, 270.0, 0], ["v000047", 228.201, 218.4, 270.0, 0], ["v000051", 218.4, 211.799, 0.0, 0], ["v000054", 218.4, 204.298, 0.0, 0], ["v000055", 235.702, 218.4, 270.0, 0], ["v000057", 613.495592, 221.6, 90.0, 0], ["v000058", 258.205, 218.4, 270.0, 0], ["v000066", 221.6, 433.00915265527146, 180.0, 0], ["v000067", 677.305382, 381.6, 90.0, 0], ["v000070", 265.706, 218.4, 270.0, 0], ["v000072", 50.301000000000045, 378.4, 270.0, 0], ["v000073", 221.6, 395.5041526552714, 180.0, 0], ["v000075", 273.207, 218.4, 270.0, 0], ["v000076", 218.4, 196.797, 0.0, 0], ["v000077", 228.201, 538.4, 270.0, 0], ["v000080", 218.4, 189.296, 0.0, 0], ["v000082", 541.6, 76.12740525820348, 180.0, 0], ["v000083", 280.708, 218.4, 270.0, 0], ["v000084", 295.71000000000004, 218.4, 270.0, 0], ["v000085", 235.702, 538.4, 270.0, 0], ["v000087", 593.09356, 221.6, 90.0, 0], ["v000089", 221.6, 440.51015265527144, 180.0, 0], ["v000090", 288.20900000000006, 218.4, 270.0, 0], ["v000091", 221.6, 448.0111526552714, 180.0, 0], ["v000093", 221.6, 418.0071526552714, 180.0, 0], ["v000095", 243.203, 538.4, 270.0, 0], ["v000096", 218.4, 524.298, 0.0, 0], ["v000097", 218.4, 516.7969999999999, 0.0, 0], ["v000098", 388.201, 218.4, 270.0, 0], ["v000099", 395.702, 218.4, 270.0, 0], ["v000100", 218.4, 371.799, 0.0, 0], ["v000101", 221.6, 548.201, 180.0, 0], ["v000102", 71.69200000000004, 378.4, 270.0, 0], ["v000103", 403.203, 218.4, 270.0, 0], ["v000104", 250.704, 538.4, 270.0, 0], ["v000106", 709.699, 221.6, 90.0, 0], ["v000109", 410.704, 218.4, 270.0, 0], ["v000110", 541.6, 104.06536525820351, 180.0, 0], ["v000111", 657.1036438, 381.6, 90.0, 0], ["v000112", 273.207, 538.4, 270.0, 0], ["v000113", 258.205, 538.4, 270.0, 0], ["v000115", 221.6, 403.00515265527144, 180.0, 0], ["v000116", 218.4, 479.7191583708982, 0.0, 0], ["v000118", 531.799, 541.6, 90.0, 0], ["v000119", 265.706, 538.4, 270.0, 0], ["v000120", 218.4, 364.298, 0.0, 0], ["v000121", 418.20500000000004, 218.4, 270.0, 0], ["v000122", 221.6, 555.702, 180.0, 0], ["v000123", 218.4, 454.2513978708982, 0.0, 0], ["v000124", 371.799, 221.6, 90.0, 0], ["v000125", 218.4, 181.795, 0.0, 0], ["v000126", 288.20900000000006, 538.4, 270.0, 0], ["v000128", 295.71000000000004, 538.4, 270.0, 0], ["v000130", 303.21100000000007, 538.4, 270.0, 0], ["v000131", 633.897624, 221.6, 90.0, 0], ["v000132", 310.71200000000005, 538.4, 270.0, 0], ["v000133", 218.4, 356.797, 0.0, 0], ["v000134", 150.2131947417965, 378.4, 270.0, 0], ["v000135", 221.6, 410.5061526552714, 180.0, 0], ["v000137", 218.4, 349.296, 0.0, 0], ["v000138", 228.201, 378.4, 270.0, 0], ["v000139", 235.702, 378.4, 270.0, 0], ["v000140", 318.2130000000001, 538.4, 270.0, 0], ["v000141", 218.4, 174.29399999999998, 0.0, 0], ["v000142", 243.203, 378.4, 270.0, 0], ["v000143", 325.71400000000006, 538.4, 270.0, 0], ["v000144", 218.4, 166.79299999999998, 0.0, 0], ["v000145", 425.706, 218.4, 270.0, 0], ["v000146", 388.201, 538.4, 270.0, 0], ["v000147", 538.4, 570.6104415, 0.0, 0], ["v000150", 371.799, 541.6, 90.0, 0], ["v000153", 433.20700000000005, 218.4, 270.0, 0], ["v000154", 218.4, 341.795, 0.0, 0], ["v000155", 388.201, 378.4, 270.0, 0], ["v000156", 364.298, 221.6, 90.0, 0], ["v000157", 440.708, 218.4, 270.0, 0], ["v000158", 303.21100000000007, 218.4, 270.0, 0], ["v000159", 395.702, 538.4, 270.0, 0], ["v000160", 448.209, 218.4, 270.0, 0], ["v000161", 395.702, 378.4, 270.0, 0], ["v000162", 403.203, 378.4, 270.0, 0], ["v000163", 455.71000000000004, 218.4, 270.0, 0], ["v000164", 403.203, 538.4, 270.0, 0], ["v000165", 221.6, 563.2030000000001, 180.0, 0], ["v000166", 531.799, 381.6, 90.0, 0], ["v000167", 470.71200000000005, 218.4, 270.0, 0], ["v000168", 364.298, 541.6, 90.0, 0], ["v000169", 520.8746666666666, 381.6, 90.0, 0], ["v000170", 505.4503333333333, 381.6, 90.0, 0], ["v000171", 463.211, 218.4, 270.0, 0], ["v000172", 709.699, 381.6, 90.0, 0], ["v000173", 218.4, 334.294, 0.0, 0], ["v000174", 572.3244558125, 221.6, 90.0, 0], ["v000175", 410.704, 538.4, 270.0, 0], ["v000176", 482.99899229999994, 381.6, 90.0, 0], ["v000177", 211.799, 381.6, 90.0, 0], ["v000178", 410.704, 378.4, 270.0, 0], ["v000179", 218.4, 326.793, 0.0, 0], ["v000180", 541.6, 388.201, 180.0, 0], ["v000181", 204.298, 381.6, 90.0, 0], ["v000182", 418.20500000000004, 378.4, 270.0, 0], ["v000183", 480.45899999999995, 541.6, 90.0, 0], ["v000184", 425.706, 378.4, 270.0, 0], ["v000185", 218.4, 319.292, 0.0, 0], ["v000186", 458.7040776999999, 381.6, 90.0, 0], ["v000187", 196.797, 381.6, 90.0, 0], ["v000188", 221.6, 570.7040000000001, 180.0, 0], ["v000189", 552.0120143125, 221.6, 90.0, 0], ["v000190", 381.6, 228.201, 180.0, 0], ["v000191", 221.6, 578.205, 180.0, 0], ["v000192", 433.20700000000005, 538.4, 270.0, 0], ["v000193", 418.20500000000004, 538.4, 270.0, 0], ["v000194", 433.20700000000005, 378.4, 270.0, 0], ["v000195", 310.71200000000005, 218.4, 270.0, 0], ["v000196", 189.296, 381.6, 90.0, 0], ["v000197", 181.795, 381.6, 90.0, 0], ["v000198", 218.4, 311.79099999999994, 0.0, 0], ["v000199", 440.708, 378.4, 270.0, 0], ["v000200", 221.6, 228.201, 180.0, 0], ["v000201", 174.29399999999998, 381.6, 90.0, 0], ["v000202", 425.706, 538.4, 270.0, 0], ["v000203", 548.201, 538.4, 270.0, 0], ["v000204", 221.6, 585.706, 180.0, 0], ["v000205", 221.6, 235.702, 180.0, 0], ["v000206", 218.4, 304.28999999999996, 0.0, 0], ["v000207", 448.209, 378.4, 270.0, 0], ["v000208", 221.6, 243.203, 180.0, 0], ["v000209", 478.213, 218.4, 270.0, 0], ["v000210", 166.79299999999998, 381.6, 90.0, 0], ["v000211", 378.4, 618.8580000000001, 0.0, 0], ["v000212", 159.29199999999997, 381.6, 90.0, 0], ["v000213", 438.21606659999986, 381.6, 90.0, 0], ["v000214", 455.71000000000004, 378.4, 270.0, 0], ["v000215", 538.4, 550.298, 0.0, 0], ["v000216", 555.702, 538.4, 270.0, 0], ["v000217", 563.2030000000001, 538.4, 270.0, 0], ["v000218", 151.79099999999997, 381.6, 90.0, 0], ["v000219", 538.4, 532.3969999999999, 0.0, 0], ["v000220", 570.7040000000001, 538.4, 270.0, 0], ["v000221", 378.4, 595.796033, 0.0, 0], ["v000222", 548.201, 218.4, 270.0, 0], ["v000223", 218.4, 159.29199999999997, 0.0, 0], ["v000224", 463.211, 378.4, 270.0, 0], ["v000225", 578.205, 538.4, 270.0, 0], ["v000226", 221.6, 388.0031526552714, 180.0, 0], ["v000227", 400.03047999999995, 541.6, 90.0, 0], ["v000228", 221.6, 508.991, 180.0, 0], ["v000229", 355.5346673, 221.6, 90.0, 0], ["v000230", 144.28999999999996, 381.6, 90.0, 0], ["v000231", 538.4, 211.799, 0.0, 0], ["v000232", 382.09515292105294, 540.2970917526317, 40.454708298894424, 0], ["v000233", 555.702, 218.4, 270.0, 0], ["v000234", 470.71200000000005, 378.4, 270.0, 0], ["v000235", 585.706, 538.4, 270.0, 0], ["v000236", 221.6, 531.92904, 172.21329861811265, 0], ["v000237", 212.39699999999996, 541.6, 90.0, 0], ["v000238", 440.708, 538.4, 270.0, 0], ["v000239", 197.09599999999998, 541.6, 90.0, 0], ["v000240", 318.2130000000001, 218.4, 270.0, 0], ["v000241", 218.4, 151.79099999999997, 0.0, 0], ["v000242", 218.4, 144.28999999999996, 0.0, 0], ["v000243", 184.39499999999998, 541.6, 90.0, 0], ["v000244", 381.6, 235.702, 180.0, 0], ["v000245", 337.5584101333333, 221.6, 90.0, 0], ["v000246", 221.6, 593.2070000000001, 180.0, 0], ["v000247", 381.6, 243.203, 180.0, 0], ["v000248", 218.4, 295.74609626666665, 0.0, 0], ["v000249", 448.209, 538.4, 270.0, 0], ["v000250", 378.4, 294.758712, 0.0, 0], ["v000251", 218.4, 136.78899999999996, 0.0, 0], ["v000252", 541.6, 395.702, 180.0, 0], ["v000253", 218.4, 129.28799999999995, 0.0, 0], ["v000254", 136.78899999999996, 381.6, 90.0, 0], ["v000255", 221.6, 600.7080000000001, 180.0, 0], ["v000256", 218.4, 121.78699999999995, 0.0, 0], ["v000257", 280.708, 538.4, 270.0, 0], ["v000258", 538.4, 204.298, 0.0, 0], ["v000259", 378.4, 274.36556959999996, 0.0, 0], ["v000260", 455.71000000000004, 538.4, 270.0, 0], ["v000261", 541.6, 403.203, 180.0, 0], ["v000262", 218.4, 114.28599999999994, 0.0, 0], ["v000263", 221.6, 608.2090000000001, 180.0, 0], ["v000264", 381.6, 548.201, 180.0, 0], ["v000265", 218.4, 277.1692647333333, 0.0, 0], ["v000266", 563.2030000000001, 218.4, 270.0, 0], ["v000267", 218.4, 106.78499999999993, 0.0, 0], ["v000268", 381.6, 555.702, 180.0, 0], ["v000269", 541.6, 410.704, 180.0, 0], ["v000270", 221.6, 615.71, 180.0, 0], ["v000271", 218.4, 99.28399999999993, 0.0, 0], ["v000272", 221.6, 623.211, 180.0, 0], ["v000273", 381.6, 563.2030000000001, 180.0, 0], ["v000274", 381.6, 570.7040000000001, 180.0, 0], ["v000275", 538.4, 196.797, 0.0, 0], ["v000276", 221.6, 630.7120000000001, 180.0, 0], ["v000277", 221.6, 638.2130000000001, 180.0, 0], ["v000278", 381.6, 578.205, 180.0, 0], ["v000279", 129.28799999999995, 381.6, 90.0, 0], ["v000280", 463.211, 538.4, 270.0, 0], ["v000281", 378.4, 253.97242719999997, 0.0, 0], ["v000282", 218.4, 91.78299999999993, 0.0, 0], ["v000283", 538.4, 189.296, 0.0, 0], ["v000284", 470.71200000000005, 538.4, 270.0, 0], ["v000285", 478.213, 538.4, 270.0, 0], ["v000286", 378.4, 233.57928479999995, 0.0, 0], ["v000287", 485.714, 538.4, 270.0, 0], ["v000288", 593.2070000000001, 538.4, 270.0, 0], ["v000289", 538.4, 181.795, 0.0, 0], ["v000290", 221.6, 645.714, 180.0, 0], ["v000291", 378.4, 213.18614239999994, 0.0, 0], ["v000292", 600.7080000000001, 538.4, 270.0, 0], ["v000293", 538.4, 174.29399999999998, 0.0, 0], ["v000294", 381.6, 585.706, 180.0, 0], ["v000295", 218.4, 84.28199999999993, 0.0, 0], ["v000296", 381.6, 593.2070000000001, 180.0, 0], ["v000297", 538.4, 166.79299999999998, 0.0, 0], ["v000298", 221.6, 653.2150000000001, 180.0, 0], ["v000299", 218.4, 76.78099999999992, 0.0, 0], ["v000300", 493.21500000000003, 538.4, 270.0, 0], ["v000301", 221.6, 660.7160000000001, 180.0, 0], ["v000302", 218.4, 69.27999999999992, 0.0, 0], ["v000303", 211.799, 221.6, 90.0, 0], ["v000304", 570.7040000000001, 218.4, 270.0, 0], ["v000305", 541.6, 548.201, 180.0, 0], ["v000306", 541.6, 555.702, 180.0, 0], ["v000307", 221.6, 668.2170000000001, 180.0, 0], ["v000308", 378.4, 192.79299999999992, 0.0, 0], ["v000309", 381.6, 600.7080000000001, 180.0, 0], ["v000310", 221.6, 675.7180000000001, 180.0, 0], ["v000311", 538.4, 159.29199999999997, 0.0, 0], ["v000312", 221.6, 683.219, 180.0, 0], ["v000313", 378.4, 174.89199999999994, 0.0, 0], ["v000314", 381.6, 608.2090000000001, 180.0, 0], ["v000315", 221.6, 690.7200000000001, 180.0, 0], ["v000316", 378.4, 159.59099999999995, 0.0, 0], ["v000317", 538.4, 151.79099999999997, 0.0, 0], ["v000318", 541.6, 563.2030000000001, 180.0, 0], ["v000319", 381.6, 615.71, 180.0, 0], ["v000320", 381.6, 623.211, 180.0, 0], ["v000321", 221.6, 698.2210000000001, 180.0, 0], ["v000322", 538.4, 144.28999999999996, 0.0, 0], ["v000323", 541.6, 570.7040000000001, 180.0, 0], ["v000324", 578.205, 218.4, 270.0, 0], ["v000325", 221.6, 705.7220000000001, 180.0, 0], ["v000326", 221.6, 713.2230000000001, 180.0, 0], ["v000327", 378.4, 146.88999999999996, 0.0, 0], ["v000328", 378.4, 136.78899999999996, 0.0, 0], ["v000329", 221.6, 720.724, 180.0, 0], ["v000330", 538.4, 136.78899999999996, 0.0, 0], ["v000331", 585.706, 218.4, 270.0, 0], ["v000332", 608.2090000000001, 538.4, 270.0, 0], ["v000333", 581.8128208, 378.4, 270.0, 0], ["v000334", 378.4, 129.28799999999995, 0.0, 0], ["v000335", 218.4, 61.77899999999992, 0.0, 0], ["v000336", 541.6, 616.0, 180.0, 0], ["v000337", 630.6863137, 538.4, 270.0, 0], ["v000338", 541.6, 643.78, 180.0, 0], ["v000339", 218.4, 54.27799999999991, 0.0, 0], ["v000340", 378.4, 102.33000000000001, 0.0, 0], ["v000341", 221.6, 728.2250000000001, 180.0, 0], ["v000342", 541.6, 664.4248842592592, 180.0, 0], ["v000343", 218.4, 46.77699999999992, 0.0, 0], ["v000344", 88.44000000000001, 221.6, 90.0, 0], ["v000345", 538.4, 88.44000000000001, 0.0, 0], ["v000346", 218.4, 39.27599999999992, 0.0, 0], ["v000347", 538.4, 68.3754279835391, 0.0, 0], ["v000348", 538.4, 48.93432098765432, 0.0, 0], ["v000349", 699.34, 378.4, 270.0, 0], ["v000350", 378.4, 60.660000000000004, 0.0, 0], ["v000351", 218.4, 27.383333333333283, 0.0, 0], ["v000352", 541.6, 741.035002, 180.0, 0], ["v000353", 18.0090882, 541.6, 90.0, 0], ["v000354", 381.6, 754.9, 180.0, 0], ["v000355", 538.4, 5.1, 0.0, 0]], "signals": [], "sensors": {}, "metrics": {"requested": 300, "total_requests": 400, "arrived": 75, "active": 280, "pending": 0, "halted": 40, "mean_system_so_far_s": 150, "collisions": 0, "teleports": 0}, "history": [[5, 5, 0, 0], [50, 20, 0, 5], [100, 40, 0, 20]], "message": ""}}', 'README.md': "# Adaptive Signals — Stage 2\n\nThis update adds a genuine live SUMO viewer, a short-horizon C++ controller,\nturn-calibrated pressure, native SUMO actuated control, separate feature ablations,\nbroader demand loads and longer draining. It extends the Stage-1 project;\nStage-1 artifacts remain separate.\n\n## Watch a live simulation\n\nOpen `AdaptiveSignals_Stage2_Live_Colab.ipynb` in Colab and run the first two code\ncells. The second cell shows the live interface inside the notebook. Click\n**Start / reset run**. Pause, resume, advance by one simulation second, change\nspeed, inspect an intersection, or reset with another controller/traffic seed.\nKeep the runtime connected. The browser renders actual positions and signal\nstates from SUMO; it does not move vehicles with a separate JavaScript model.\n\nThe viewer is a local runtime application, not a persistently hosted website.\nColab uses `serve_kernel_port_as_iframe`, following Google's current helper;\nthe deprecated window helper is not used. No public tunnel is required.\nThe iframe/proxy browser workflow has not been exercised here. The local HTTP\nAPI and real SUMO pause/step/reset/stop behavior are tested automatically.\n\nFor a local Linux CPU environment with g++:\n\n```bash\npython -m pip install -r requirements.txt\npython -m unittest discover -s tests -v\npython live_server.py --port 8765\n```\n\nOpen `http://127.0.0.1:8765/`. The server binds to loopback. Ctrl+C closes it.\nThe HTTP handler never accesses libsumo; one worker owns the simulation and\npublishes snapshots. The interface polls at 5 Hz and uses a per-process access\nkey. Controller settings take effect on reset. Detector overlays show the\ncontroller's observations; counters and vehicle positions show simulator truth.\n\n`runs/demo/traffic_comparison.gif` is a **recorded comparison**, not the live\ninterface. Its vehicles come from actual SUMO trajectories for a separate demo\nseed. It is supplied so the difference can be viewed without starting Colab.\n\n## Rerun the extended benchmark\n\n```bash\npython stage2.py --output runs/my_stage2\npython stage2.py --output runs/my_stage2 --resume\n```\n\nThe full campaign has 48 development runs and 284 evaluation runs. A short\npipeline check is available with `--profile smoke --output runs/my_smoke`.\nThe notebook's optional third code cell can rerun either profile and export\nthe source and completed benchmark results. It is disabled by default so the\nlive viewer opens without waiting for the full campaign.\n\nExperiments use 900 s of arriving traffic, then up to 1800 s without new requests.\nA run may stop early only after all requested vehicles arrive. The primary score\nincludes all requests and is still capped at the common 2700 s cutoff for runs\nthat do not clear. A cleared run has exact completed system time, independent of\nwhen it exits early. Do not compare Stage-1 scores directly to these scores:\ndrain limits, seeds and fixed timing selection differ.\n\n## What changed in the controllers\n\n- Fixed greens are selected separately for each load multiplier using development\n  data from candidates 10, 20, 30 and 45 s.\n- Native SUMO actuated control is a separate, activated program with genuine\n  gap-dependent green times. It uses built-in loop detectors and is tested only\n  in the perfect-detector comparisons, not as if it used corrupted camera counts.\n- Turn-calibrated pressure learns turning fractions from development route files\n  only; no evaluation vehicle route is exposed to the controller.\n- Receiving-aware, message-only, moving-only and Stage-1 coordinated variants\n  separate feature contributions using equal turning weights.\n- Discharge lookahead estimates waiting-time benefit from a short queue release,\n  discounts unavailable downstream capacity, and charges switching lost time.\n  It uses development-calibrated turning fractions. This is an approximate\n  short-horizon heuristic, not a solved optimal-control program or an RL model.\n\nAll external controllers retain protected approach stages, 3 s yellow and 2 s\nall-red. Green is bounded at 10–50 s. A common red-age target helps prevent neglect\nbut is not a formal upper bound on individual vehicle waiting.\n\n## Reproducibility and limits\n\n`EXPERIMENT_PLAN.md`, `STAGE2_REPORT.md`, `runs/stage2/selection.json`,\n`manifest.json`, `audit.json`, raw trip ledgers and signal traces document the\naccepted experiment. `diagnostics/` retains summaries from the rejected native\nreference initialization. It was found to retain static timing and was corrected\nbefore the accepted full campaign was rerun.\n\nNo real CCTV, RL training, motorcycles, pedestrians, OSM city calibration or\nfield hardware is implemented. New controller selection is exploratory over a\nsmall development grid and four evaluation seeds. These results are not evidence\nof superiority on real Thai roads or an established public benchmark.\n\nSources:\n\n- https://sumo.dlr.de/docs/Simulation/Traffic_Lights.html\n- https://sumo.dlr.de/docs/TraCI/Traffic_Lights_Value_Retrieval.html\n- https://sumo.dlr.de/docs/TraCI/Lane_Value_Retrieval.html\n- https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py\n- https://github.com/Pi-Star-Lab/RESCO\n", 'EXPERIMENT_PLAN.md': "# Stage 2 — development, evaluation and live simulation\n\n## Questions\n\n1. Does the initial coordination effect survive lighter demand and longer draining?\n2. What do receiving capacity, upstream messages and moving counts contribute?\n3. Does a short-horizon discharge heuristic improve over fixed, native actuated,\n   equal-turn pressure, turn-calibrated pressure and the Stage-1 candidate?\n4. Does it still behave under the Stage-1 delayed/noisy/missing-count regime?\n\n## Prespecified campaign\n\nThe Stage-1 synthetic 3×3 left-hand grid and protected single-approach stages are\nretained. Passenger cars have explicit speed factors and deterministic car\nfollowing. Demand multipliers are 0.3, 0.6 and 1.0 relative to Stage 1.\nDemand lasts 900 s; the common maximum drain is extended to 1800 s. A run exits\nearly only when all requests have arrived. No requests are removed or teleported.\n\nDevelopment streams: balanced ×0.3, directional switch ×0.6, surge ×1.0, each with\nseeds 201 and 1201. Fixed duration is chosen by load from [10,20,30,45] s to minimize\nmean system time on its two development streams. Turning fractions are counted\nfrom all six development route files and frozen. Unsupported incoming lanes use\nequal weights; otherwise only observed development outgoing turns have weight.\n\nFour discharge configurations are compared: (horizon,receiving power) =\n(10,1), (20,1), (30,1), (20,2). Message and moving weights remain 0.30 and 0.20.\nSelect the configuration with the lowest mean ratio of development total system\ntime to the development-selected fixed controller over the six streams.\n24 fixed + 24 discharge = 48 development runs. Save selection before evaluation.\n\nEvaluation seeds 12777,13777,14777,15777 are not used for development.\nBalanced and directional-switch scenarios are tested at all three loads, plus\nsurge ×0.6, with exact counts. Each of these seven cases has nine controllers and\nfour seeds: 252 runs. Directional-switch ×1.0 additionally has degraded counts\nwith eight controllers and four seeds: 32 runs. Total: **284 evaluations**.\nThe native actuated reference uses loop detectors, so it is omitted from the\ncorrupted-camera-count case rather than receiving privileged sensors unnoticed.\n\n## Controllers\n\nAll external controllers share the protected stages and min/max green, yellow,\nall-red and red-age-target mechanism. All decide synchronously from one snapshot.\n\n- Fixed: cyclic order, development-selected equal green duration for that load.\n- Native SUMO actuated: separate activated program, min/max greens 10/50 s,\n  3 s yellow and 2 s all-red; built-in gap-based loop-detector logic.\n- Equal-turn pressure: incoming stopped queue minus mean downstream stopped queue.\n- Turn-calibrated pressure: use development-estimated turn fractions instead.\n- Receiving-aware: equal-turn pressure multiplied by downstream free-fraction².\n- Message-only / moving-only: receiving-aware plus just the respective added term.\n- Stage-1 coordinated: both extra terms with the original coefficients.\n- Discharge lookahead: calibrated-turn queue-discharge estimate described below.\n\nFor each incoming lane let q be its stopped queue, a its upstream release\nannouncement, m its moving count, d the turn-weighted downstream stopped queue,\nand f the turn-weighted downstream occupied-capacity fraction. Let\nD=max(0,q+0.30a+0.20m). The available green horizon is T=H for holding, H−5 for\nswitching (3 s yellow + 2 s all-red). Assumed saturation rate s=0.45 vehicles/s.\nEstimated released count is R=min(D,sT). Score:\n\n`(R*T - R*R/(2*s)) * (1-f)^p - 0.25*R*d`\n\nThe first term approximates saved stopped waiting during a release, receiving\ncapacity discounts it, and the last term penalizes feeding downstream queues.\nThis is deliberately called a heuristic; it is not a calibrated queue simulator,\na formal model-predictive optimization or a new theoretical result. Phase choice\nalso obeys hysteresis and the common red-age target. Its comparisons change\nturning weights as well as score, so those contributions are not fully isolated.\nThe numeric hysteresis is 2 in each controller's own score units; it is not a\nphysically equal threshold across queue-pressure and waiting-benefit scores.\n\n## Measurements and safeguards\n\nPrimary: total system time per requested vehicle, including insertion waiting\nand unfinished journeys at the common maximum horizon. Secondary: completion,\ncleared flag, actual finish time, stopped waiting, backlog, time loss, switches\nand runtime. Keep completed-trip percentiles secondary because of survivor bias.\n\nRecheck conservation, metric against the independent SUMO trip ledger,\nmatching controller input hashes, clearance transitions, min/max green and\ncollision/teleport events. Save invalid runs and fail instead of silently excluding\nthem. Resume requires identical source/configuration/version and input hashes.\n\nThe native reference's first implementation changed the type of an existing\nstatic program. Runtime signal traces showed that this retained static behavior.\nIt was rejected; the corrected reference creates and activates a new actuated\nprogram. A test now requires short gap-dependent greens. The entire accepted\ncampaign is rerun with the corrected initialization, without changing controller\ncoefficients or evaluation seeds. This is an implementation repair, not new\nindependent confirmation after inspecting earlier results.\n\n## Live viewer and verification scope\n\nThe viewer uses the identical runner and C++ kernels. One worker owns libsumo;\nHTTP threads read snapshots and issue commands. Pause, one-second step, resume,\nreset, stop, sensor corruption and speed controls are runtime behavior, not\nreplayed trajectories. The supplied GIF is explicitly a separate recorded demo.\nChanging controller/demand restarts the seeded demand stream.\n\nHTTP integration tests exercise actual SUMO operation, authentication and invalid\ninput handling. JavaScript syntax and mocked-canvas numerical checks are separate\nfrom browser visual QA. Colab's private iframe helper is wired but its browser\nproxy is not claimed exercised here. No public server or field controller is\ndeployed.\n\n## Next research gate\n\nSeparate turning calibration from discharge scoring; measure actual saturation\nand validate it. Use a held-out topology and an established public benchmark.\nOnly then add RL training with a distinct evaluation split. A Thai case study\nrequires road, lane, turn, signal and measured-demand audits; OSM geometry alone\ndoes not supply those quantities.\n", 'STAGE2_REPORT.md': '# Stage 2 — results and live simulation\n\n**Accepted campaign: 48 development runs and 284 evaluation runs.** All accepted evaluations pass conservation and independent metric-ledger checks, with zero collision and teleport events. The selected discharge controller clears all requests in all 32 of its evaluated runs.\n\nThe selected controller uses a 20 s discharge horizon, receiving power 1, upstream-message weight 0.30 and moving-count weight 0.20. Selection was made on six development streams using seeds 201/1201 before evaluation. Evaluation uses seeds 12777/13777/14777/15777. Fixed greens selected by load are 30 s (×0.3), 20 s (×0.6), and 20 s (×1.0).\n\n## Main comparison\n\nScores below are finite-horizon mean system time per requested vehicle. The cutoff is 2700 s (900 s arrivals + up to 1800 s draining). Requests waiting to enter and unfinished trips remain counted. Early exit occurs only after every request arrives. All discharge runs clear before the cutoff; some baseline runs do not. Percentage reductions are matched within seed and averaged over four seeds.\n\n| Demand | Load | Sensors | Fixed (s/request) | Native actuated (s/request) | Discharge (s/request) | Paired reduction vs fixed | Discharge cleared |\n|---|---:|---|---:|---:|---:|---:|---:|\n| balanced | 0.3 | perfect | 211.3 | 163.7 | 132.6 | 37.2% | 4/4 |\n| balanced | 0.6 | perfect | 442.1 | 361.7 | 241.7 | 45.3% | 4/4 |\n| balanced | 1 | perfect | 911.2 | 1019.1 | 506.1 | 44.4% | 4/4 |\n| directional_switch | 0.3 | perfect | 235.0 | 165.4 | 135.2 | 42.5% | 4/4 |\n| directional_switch | 0.6 | perfect | 544.2 | 474.9 | 313.8 | 42.4% | 4/4 |\n| directional_switch | 1 | degraded | 1140.3 | not comparable | 659.6 | 42.1% | 4/4 |\n| directional_switch | 1 | perfect | 1140.3 | 1113.8 | 696.3 | 38.9% | 4/4 |\n| surge | 0.6 | perfect | 770.7 | 687.3 | 430.8 | 44.1% | 4/4 |\n\nAcross these cases, discharge reduces mean matched system time by about 37–45% relative to the development-tuned fixed baseline. This is a result on this synthetic grid, not a real-city reduction estimate. It is also better on mean system time than the corrected native actuated reference in the seven exact-count comparisons; all raw controller results are retained. The native reference is omitted from the corrupted-camera case because it uses separate loop detectors.\n\n## What the additional tests reveal\n\n- Extending the drain changes the interpretation of heavy traffic. The new candidate clears every evaluated demand stream, while several baselines retain a substantial unresolved backlog. Stage-1 percentages are not directly comparable because its cutoff was shorter and its evaluation seeds/timing selection differed.\n- Turn calibration alone helps in some light-demand cases but is weak in congested cases. Knowing turn fractions does not by itself solve downstream blocking.\n- The receiving-capacity and coordination variants are useful intermediate comparisons. Message-only and moving-only ablations are included; their ordering varies with traffic. Adding features does not automatically improve every case.\n- Noisy/delayed/missing counts do not eliminate the candidate advantage in the tested heavy directional-switch case. They sometimes produce a better score than exact counts; that is not evidence that sensor failures are beneficial.\n- The discharge score uses turn calibration plus a changed score and switching-loss treatment. Their separate contributions are not yet fully isolated.\n\n## Live simulation\n\nThe live viewer uses the same runner and C++ controllers, not a JavaScript replacement model. It displays measured vehicle positions, current signals, approach queue estimates, completed trips, active vehicles, insertion backlog and system time so far. Controls include start/reset, pause, resume, one-second step, speed, traffic pattern, load, seed and detector corruption. The same seed regenerates identical requests when switching controller.\n\nRun the first two code cells of `AdaptiveSignals_Stage2_Live_Colab.ipynb`, then press Start / reset run. Keep the runtime connected. The optional third cell reruns the benchmark and exports evidence. Locally, run `python live_server.py` and open the printed loopback address. This is a runtime application, not a published website.\n\nThe supplied GIF is explicitly a recorded three-controller demonstration using demo seed 22777, separate from evaluation. It can be watched immediately but is not the live UI and is not an additional benchmark.\n\n## Verification and limits\n\n- Ten Python tests pass, including real HTTP start/pause/step/reset/stop, access-key checks, invalid-input rejection, native-actuated variable timing, metric conservation and signal clearance.\n- Viewer JavaScript passes syntax validation and a mocked-DOM/canvas numerical smoke check using SUMO-shaped data. This is not a browser rendering test.\n- The standalone notebook cells are validated in a fresh project directory, including live server startup and the short benchmark profile. The actual Colab browser iframe/proxy has not been exercised here.\n- Resume is verified under matching source/configuration and input hashes. Each of 32 evaluation groups shares exact demand and network identities across controllers.\n- The first native-reference initialization retained static control despite changing its reported type. That experiment was rejected; the corrected reference creates and activates a distinct actuated program. The full 332-run campaign was rerun. Diagnostic summaries/logs remain available. Evaluation seeds were already exposed during the repair, so the accepted rerun is a corrected exploratory experiment, not independent confirmatory evidence.\n- There is no RL training, real CCTV access, real Thai traffic calibration, pedestrian/motorcycle model or field deployment. Four seeds on one synthetic topology cannot establish research novelty or broad superiority. Baselines retain default/common signal assumptions; no coordinated fixed-time offset optimization or reference public benchmark is included.\n\n## Next optimization gate\n\nValidate on a new topology and a public traffic-control benchmark. Separate turning calibration, switching-loss treatment and the discharge formula. Calibrate saturation flow rather than fixing 0.45 vehicles/s. Add pedestrians and mixed vehicles only with appropriate model checks. Then introduce RL with a separate training/evaluation split, preserving the current strong non-learning controllers.\n', 'runs/stage2/selection.json': '{\n  "fixed_green_by_load": {\n    "0.3": 30,\n    "0.6": 20,\n    "1.0": 20\n  },\n  "predictive_config": {\n    "horizon_s": 20,\n    "receiving_power": 1,\n    "message_weight": 0.3,\n    "moving_weight": 0.2\n  },\n  "candidate_scores": [\n    0.614590424496487,\n    0.6061786467216299,\n    0.6253909382503746,\n    0.629422022537437\n  ],\n  "objective": "mean ratio of total system time to dev-selected fixed timing across six dev streams",\n  "selected_before_evaluation": true,\n  "turn_weight_sha256": "3506d0bbff2dabe415237803789951bfad6235ed1c7e85df12fb515a582e13f5"\n}\n', 'runs/stage2/turn_weights.json': '{\n  "A0A1_0": {\n    "A1A2_0": 0.7989457831325302,\n    "A1B1_0": 0.10316265060240964,\n    "A1left1_0": 0.09789156626506024\n  },\n  "A0B0_0": {\n    "B0C0_0": 0.8053691275167785,\n    "B0bottom1_0": 0.19463087248322147\n  },\n  "A1A0_0": {\n    "A0B0_0": 0.43717472118959105,\n    "A0bottom0_0": 0.3509293680297398,\n    "A0left0_0": 0.21189591078066913\n  },\n  "A1A2_0": {\n    "A2B2_0": 0.44766146993318484,\n    "A2left2_0": 0.2301410541945063,\n    "A2top0_0": 0.32219747587230885\n  },\n  "A1B1_0": {\n    "B1C1_0": 1.0\n  },\n  "A2A1_0": {\n    "A1A0_0": 0.7769461077844312,\n    "A1B1_0": 0.10329341317365269,\n    "A1left1_0": 0.11976047904191617\n  },\n  "A2B2_0": {\n    "B2C2_0": 0.7736842105263158,\n    "B2top1_0": 0.22631578947368422\n  },\n  "B0A0_0": {\n    "A0A1_0": 0.8330914368650217,\n    "A0left0_0": 0.16690856313497823\n  },\n  "B0B1_0": {\n    "B1B2_0": 1.0\n  },\n  "B0C0_0": {\n    "C0bottom2_0": 0.40324763193504737,\n    "C0right0_0": 0.5967523680649527\n  },\n  "B1A1_0": {\n    "A1A0_0": 0.33940774487471526,\n    "A1A2_0": 0.33029612756264237,\n    "A1left1_0": 0.33029612756264237\n  },\n  "B1B0_0": {\n    "B0C0_0": 0.31235955056179776,\n    "B0bottom1_0": 0.6876404494382022\n  },\n  "B1B2_0": {\n    "B2C2_0": 0.35864978902953587,\n    "B2top1_0": 0.6413502109704642\n  },\n  "B1C1_0": {\n    "C1right1_0": 1.0\n  },\n  "B2A2_0": {\n    "A2A1_0": 0.8143236074270557,\n    "A2left2_0": 0.1856763925729443\n  },\n  "B2B1_0": {\n    "B1B0_0": 1.0\n  },\n  "B2C2_0": {\n    "C2right2_0": 0.5672823218997362,\n    "C2top2_0": 0.43271767810026385\n  },\n  "C0B0_0": {\n    "B0A0_0": 0.7854137447405329,\n    "B0B1_0": 0.21458625525946703\n  },\n  "C0C1_0": {\n    "C1C2_0": 1.0\n  },\n  "C1B1_0": {\n    "B1A1_0": 1.0\n  },\n  "C1C0_0": {\n    "C0bottom2_0": 1.0\n  },\n  "C1C2_0": {\n    "C2top2_0": 1.0\n  },\n  "C2B2_0": {\n    "B2A2_0": 0.7842876165113183,\n    "B2B1_0": 0.21571238348868177\n  },\n  "C2C1_0": {\n    "C1C0_0": 1.0\n  },\n  "bottom0A0_0": {\n    "A0A1_0": 1.0\n  },\n  "bottom1B0_0": {\n    "B0A0_0": 0.2866666666666667,\n    "B0B1_0": 0.7133333333333334\n  },\n  "bottom2C0_0": {\n    "C0B0_0": 0.714622641509434,\n    "C0C1_0": 0.28537735849056606\n  },\n  "left0A0_0": {\n    "A0A1_0": 0.6314553990610329,\n    "A0B0_0": 0.3685446009389671\n  },\n  "left1A1_0": {\n    "A1A0_0": 0.35909090909090907,\n    "A1A2_0": 0.32045454545454544,\n    "A1B1_0": 0.32045454545454544\n  },\n  "left2A2_0": {\n    "A2A1_0": 0.6270783847980997,\n    "A2B2_0": 0.37292161520190026\n  },\n  "right0C0_0": {\n    "C0B0_0": 1.0\n  },\n  "right1C1_0": {\n    "C1B1_0": 1.0\n  },\n  "right2C2_0": {\n    "C2B2_0": 1.0\n  },\n  "top0A2_0": {\n    "A2A1_0": 1.0\n  },\n  "top1B2_0": {\n    "B2A2_0": 0.36830357142857145,\n    "B2B1_0": 0.6316964285714286\n  },\n  "top2C2_0": {\n    "C2B2_0": 0.6730769230769231,\n    "C2C1_0": 0.3269230769230769\n  }\n}\n', 'reference/stage2_aggregate.json': '[\n  {\n    "scenario": "balanced",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "calibrated_pressure",\n    "runs": 4,\n    "mean_system_time_s": 163.4035,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 22.652013652335103,\n    "min_reduction_pct": 21.87813573332006,\n    "max_reduction_pct": 23.571009347443862,\n    "mean_max_stopped_wait_s": 344.25,\n    "mean_switches": 409.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "coordinated",\n    "runs": 4,\n    "mean_system_time_s": 154.8225,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 26.70901050696576,\n    "min_reduction_pct": 24.62068108677891,\n    "max_reduction_pct": 28.316010333937662,\n    "mean_max_stopped_wait_s": 343.75,\n    "mean_switches": 379,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "fixed",\n    "runs": 4,\n    "mean_system_time_s": 211.26725,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 0.0,\n    "min_reduction_pct": 0.0,\n    "max_reduction_pct": 0.0,\n    "mean_max_stopped_wait_s": 303.5,\n    "mean_switches": 297,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "max_pressure",\n    "runs": 4,\n    "mean_system_time_s": 161.97674999999998,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 23.311968118468503,\n    "min_reduction_pct": 20.800544587246023,\n    "max_reduction_pct": 24.734270863127815,\n    "mean_max_stopped_wait_s": 314.75,\n    "mean_switches": 375.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "message_only",\n    "runs": 4,\n    "mean_system_time_s": 162.28375,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 23.167934309873818,\n    "min_reduction_pct": 21.5694225858748,\n    "max_reduction_pct": 25.49495127807973,\n    "mean_max_stopped_wait_s": 327.75,\n    "mean_switches": 364.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "moving_only",\n    "runs": 4,\n    "mean_system_time_s": 155.59375,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 26.342919247507727,\n    "min_reduction_pct": 24.62068108677891,\n    "max_reduction_pct": 27.18831025751253,\n    "mean_max_stopped_wait_s": 343.25,\n    "mean_switches": 381.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "predictive",\n    "runs": 4,\n    "mean_system_time_s": 132.58875,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 37.244534891644115,\n    "min_reduction_pct": 36.422837077209415,\n    "max_reduction_pct": 37.72479651664262,\n    "mean_max_stopped_wait_s": 168.25,\n    "mean_switches": 541.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "receiving_aware",\n    "runs": 4,\n    "mean_system_time_s": 162.02575000000002,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 23.290219877266257,\n    "min_reduction_pct": 22.12230968380387,\n    "max_reduction_pct": 25.49495127807973,\n    "mean_max_stopped_wait_s": 303.25,\n    "mean_switches": 368.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "sumo_actuated",\n    "runs": 4,\n    "mean_system_time_s": 163.688,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 22.511411485384116,\n    "min_reduction_pct": 20.99481454034106,\n    "max_reduction_pct": 23.191062633148878,\n    "mean_max_stopped_wait_s": 200.0,\n    "mean_switches": 622.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "calibrated_pressure",\n    "runs": 4,\n    "mean_system_time_s": 291.28375,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 34.05194403360521,\n    "min_reduction_pct": 31.83008814458778,\n    "max_reduction_pct": 38.06059700630939,\n    "mean_max_stopped_wait_s": 598.5,\n    "mean_switches": 613,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "coordinated",\n    "runs": 4,\n    "mean_system_time_s": 282.5485,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 36.06622132349905,\n    "min_reduction_pct": 35.17740899782479,\n    "max_reduction_pct": 38.23962780108463,\n    "mean_max_stopped_wait_s": 593.75,\n    "mean_switches": 582.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "fixed",\n    "runs": 4,\n    "mean_system_time_s": 442.07975,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 0.0,\n    "min_reduction_pct": 0.0,\n    "max_reduction_pct": 0.0,\n    "mean_max_stopped_wait_s": 1154.25,\n    "mean_switches": 715.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "max_pressure",\n    "runs": 4,\n    "mean_system_time_s": 300.48275,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 32.0269595094558,\n    "min_reduction_pct": 30.582506694241314,\n    "max_reduction_pct": 32.86934679730621,\n    "mean_max_stopped_wait_s": 596.25,\n    "mean_switches": 619.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "message_only",\n    "runs": 4,\n    "mean_system_time_s": 287.54775,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 34.92243645893383,\n    "min_reduction_pct": 32.63068938143294,\n    "max_reduction_pct": 37.88317331023053,\n    "mean_max_stopped_wait_s": 559.75,\n    "mean_switches": 589,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "moving_only",\n    "runs": 4,\n    "mean_system_time_s": 282.746,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 36.0323488997989,\n    "min_reduction_pct": 33.80691489761695,\n    "max_reduction_pct": 38.405480386549655,\n    "mean_max_stopped_wait_s": 573.75,\n    "mean_switches": 571,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "predictive",\n    "runs": 4,\n    "mean_system_time_s": 241.747,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 45.26779624650747,\n    "min_reduction_pct": 43.39089488222299,\n    "max_reduction_pct": 48.57712945559274,\n    "mean_max_stopped_wait_s": 524.0,\n    "mean_switches": 657.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "receiving_aware",\n    "runs": 4,\n    "mean_system_time_s": 284.4385,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 35.65191589752861,\n    "min_reduction_pct": 32.653525589157596,\n    "max_reduction_pct": 37.40490073819185,\n    "mean_max_stopped_wait_s": 561.0,\n    "mean_switches": 586.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "sumo_actuated",\n    "runs": 4,\n    "mean_system_time_s": 361.695,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 18.573944257154025,\n    "min_reduction_pct": 7.065361238106623,\n    "max_reduction_pct": 38.71396726415372,\n    "mean_max_stopped_wait_s": 905.0,\n    "mean_switches": 734.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "calibrated_pressure",\n    "runs": 4,\n    "mean_system_time_s": 1123.0774999999999,\n    "mean_completion": 0.9433181337327792,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": -23.244537722764825,\n    "min_reduction_pct": -41.279044516106225,\n    "max_reduction_pct": -13.98951884394592,\n    "mean_max_stopped_wait_s": 2003.5,\n    "mean_switches": 1149.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "coordinated",\n    "runs": 4,\n    "mean_system_time_s": 727.1972499999999,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 20.18674287774216,\n    "min_reduction_pct": 17.81607656964932,\n    "max_reduction_pct": 23.606330364288436,\n    "mean_max_stopped_wait_s": 1237.75,\n    "mean_switches": 955.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "fixed",\n    "runs": 4,\n    "mean_system_time_s": 911.1912500000001,\n    "mean_completion": 0.9232510505137119,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": 0.0,\n    "min_reduction_pct": 0.0,\n    "max_reduction_pct": 0.0,\n    "mean_max_stopped_wait_s": 2214.0,\n    "mean_switches": 963,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "max_pressure",\n    "runs": 4,\n    "mean_system_time_s": 1562.62925,\n    "mean_completion": 0.38914482383304705,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": -71.52744459279288,\n    "min_reduction_pct": -80.99650393303511,\n    "max_reduction_pct": -63.639115521514114,\n    "mean_max_stopped_wait_s": 2447.25,\n    "mean_switches": 942.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "message_only",\n    "runs": 4,\n    "mean_system_time_s": 736.53125,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 19.17318500522496,\n    "min_reduction_pct": 12.43289721220327,\n    "max_reduction_pct": 21.69439909021888,\n    "mean_max_stopped_wait_s": 1211.0,\n    "mean_switches": 977.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "moving_only",\n    "runs": 4,\n    "mean_system_time_s": 723.15225,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 20.60207681644338,\n    "min_reduction_pct": 16.81645713389737,\n    "max_reduction_pct": 23.49377226023763,\n    "mean_max_stopped_wait_s": 1197.75,\n    "mean_switches": 938.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "predictive",\n    "runs": 4,\n    "mean_system_time_s": 506.09325,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 44.44329149320181,\n    "min_reduction_pct": 40.81282747719896,\n    "max_reduction_pct": 47.79561304467873,\n    "mean_max_stopped_wait_s": 1151.75,\n    "mean_switches": 920.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "receiving_aware",\n    "runs": 4,\n    "mean_system_time_s": 758.9105,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 16.68720198192473,\n    "min_reduction_pct": 7.78932615950857,\n    "max_reduction_pct": 22.335391500817664,\n    "mean_max_stopped_wait_s": 1251.25,\n    "mean_switches": 968,\n    "all_valid": true\n  },\n  {\n    "scenario": "balanced",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "sumo_actuated",\n    "runs": 4,\n    "mean_system_time_s": 1019.11525,\n    "mean_completion": 0.9875333422911275,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": -11.752185074836659,\n    "min_reduction_pct": -17.606006654154612,\n    "max_reduction_pct": -6.065597045996821,\n    "mean_max_stopped_wait_s": 2049.5,\n    "mean_switches": 905.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "calibrated_pressure",\n    "runs": 4,\n    "mean_system_time_s": 163.38225,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 30.464359001505077,\n    "min_reduction_pct": 28.636363530015498,\n    "max_reduction_pct": 31.505173599956663,\n    "mean_max_stopped_wait_s": 310.75,\n    "mean_switches": 421.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "coordinated",\n    "runs": 4,\n    "mean_system_time_s": 157.0075,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 33.170748050412975,\n    "min_reduction_pct": 31.139924671306826,\n    "max_reduction_pct": 34.48795905397194,\n    "mean_max_stopped_wait_s": 307.5,\n    "mean_switches": 403.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "fixed",\n    "runs": 4,\n    "mean_system_time_s": 234.98125,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 0.0,\n    "min_reduction_pct": 0.0,\n    "max_reduction_pct": 0.0,\n    "mean_max_stopped_wait_s": 338.5,\n    "mean_switches": 317.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "max_pressure",\n    "runs": 4,\n    "mean_system_time_s": 162.2865,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 30.92375150881815,\n    "min_reduction_pct": 29.43503805785156,\n    "max_reduction_pct": 32.61544150665089,\n    "mean_max_stopped_wait_s": 329.75,\n    "mean_switches": 409,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "message_only",\n    "runs": 4,\n    "mean_system_time_s": 164.75125,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 29.866560005007663,\n    "min_reduction_pct": 27.29713979645867,\n    "max_reduction_pct": 32.535079258166355,\n    "mean_max_stopped_wait_s": 301.0,\n    "mean_switches": 401.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "moving_only",\n    "runs": 4,\n    "mean_system_time_s": 156.518,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 33.374162833156745,\n    "min_reduction_pct": 31.334254336126683,\n    "max_reduction_pct": 35.568754138457535,\n    "mean_max_stopped_wait_s": 305.25,\n    "mean_switches": 399.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "predictive",\n    "runs": 4,\n    "mean_system_time_s": 135.2035,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 42.456749822446056,\n    "min_reduction_pct": 41.27427285630539,\n    "max_reduction_pct": 42.93699785200154,\n    "mean_max_stopped_wait_s": 188.5,\n    "mean_switches": 542.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "receiving_aware",\n    "runs": 4,\n    "mean_system_time_s": 164.02525,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 30.176258763441364,\n    "min_reduction_pct": 27.270180767813137,\n    "max_reduction_pct": 32.535079258166355,\n    "mean_max_stopped_wait_s": 303.25,\n    "mean_switches": 400.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.3,\n    "regime": "perfect",\n    "controller": "sumo_actuated",\n    "runs": 4,\n    "mean_system_time_s": 165.44,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 29.5825673631197,\n    "min_reduction_pct": 27.3094960179212,\n    "max_reduction_pct": 31.538365237620404,\n    "mean_max_stopped_wait_s": 197.0,\n    "mean_switches": 617.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "calibrated_pressure",\n    "runs": 4,\n    "mean_system_time_s": 435.09749999999997,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 20.137438555375226,\n    "min_reduction_pct": 17.44390848584842,\n    "max_reduction_pct": 24.79884647851311,\n    "mean_max_stopped_wait_s": 829.75,\n    "mean_switches": 704.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "coordinated",\n    "runs": 4,\n    "mean_system_time_s": 407.7335,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 25.1278937199109,\n    "min_reduction_pct": 19.213318344381005,\n    "max_reduction_pct": 28.827874211262305,\n    "mean_max_stopped_wait_s": 833.5,\n    "mean_switches": 647,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "fixed",\n    "runs": 4,\n    "mean_system_time_s": 544.22925,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 0.0,\n    "min_reduction_pct": 0.0,\n    "max_reduction_pct": 0.0,\n    "mean_max_stopped_wait_s": 1271.75,\n    "mean_switches": 803.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "max_pressure",\n    "runs": 4,\n    "mean_system_time_s": 551.1614999999999,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": -1.392629897516582,\n    "min_reduction_pct": -7.092818467433145,\n    "max_reduction_pct": 2.595088345119979,\n    "mean_max_stopped_wait_s": 1033.5,\n    "mean_switches": 754.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "message_only",\n    "runs": 4,\n    "mean_system_time_s": 415.84075,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 23.62414676455894,\n    "min_reduction_pct": 21.193416806133886,\n    "max_reduction_pct": 25.779230197378165,\n    "mean_max_stopped_wait_s": 832.75,\n    "mean_switches": 660,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "moving_only",\n    "runs": 4,\n    "mean_system_time_s": 413.1505,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 24.071007543139267,\n    "min_reduction_pct": 21.734762250426915,\n    "max_reduction_pct": 26.70512206660964,\n    "mean_max_stopped_wait_s": 830.5,\n    "mean_switches": 646,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "predictive",\n    "runs": 4,\n    "mean_system_time_s": 313.7745,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 42.36805779226603,\n    "min_reduction_pct": 39.87825660312597,\n    "max_reduction_pct": 44.22751926612369,\n    "mean_max_stopped_wait_s": 679.75,\n    "mean_switches": 691,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "receiving_aware",\n    "runs": 4,\n    "mean_system_time_s": 423.87,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 22.069498455604815,\n    "min_reduction_pct": 20.201083809925922,\n    "max_reduction_pct": 24.393913167156626,\n    "mean_max_stopped_wait_s": 883.25,\n    "mean_switches": 654.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "sumo_actuated",\n    "runs": 4,\n    "mean_system_time_s": 474.93875,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 12.71320540035676,\n    "min_reduction_pct": 3.5206109476880876,\n    "max_reduction_pct": 19.89503117717428,\n    "mean_max_stopped_wait_s": 1071.0,\n    "mean_switches": 674,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "degraded",\n    "controller": "calibrated_pressure",\n    "runs": 4,\n    "mean_system_time_s": 1340.0525,\n    "mean_completion": 0.7495771649549899,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": -17.657267919868772,\n    "min_reduction_pct": -23.93846337945845,\n    "max_reduction_pct": -4.600359664281855,\n    "mean_max_stopped_wait_s": 2330.75,\n    "mean_switches": 1084,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "degraded",\n    "controller": "coordinated",\n    "runs": 4,\n    "mean_system_time_s": 961.95475,\n    "mean_completion": 0.9996264236495312,\n    "cleared_runs": 2,\n    "paired_reduction_vs_fixed_pct": 15.531309602609943,\n    "min_reduction_pct": 4.684370327313767,\n    "max_reduction_pct": 19.31203430224044,\n    "mean_max_stopped_wait_s": 1613.25,\n    "mean_switches": 980,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "degraded",\n    "controller": "fixed",\n    "runs": 4,\n    "mean_system_time_s": 1140.28525,\n    "mean_completion": 0.8287020157334446,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": 0.0,\n    "min_reduction_pct": 0.0,\n    "max_reduction_pct": 0.0,\n    "mean_max_stopped_wait_s": 2178.5,\n    "mean_switches": 963,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "degraded",\n    "controller": "max_pressure",\n    "runs": 4,\n    "mean_system_time_s": 1668.84525,\n    "mean_completion": 0.3895853583192377,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": -46.45575276441374,\n    "min_reduction_pct": -51.17207672806442,\n    "max_reduction_pct": -41.264537568471056,\n    "mean_max_stopped_wait_s": 2501.25,\n    "mean_switches": 1069,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "degraded",\n    "controller": "message_only",\n    "runs": 4,\n    "mean_system_time_s": 966.816,\n    "mean_completion": 0.9990701284309111,\n    "cleared_runs": 1,\n    "paired_reduction_vs_fixed_pct": 15.12153719089413,\n    "min_reduction_pct": 10.180125772301833,\n    "max_reduction_pct": 18.95256021639402,\n    "mean_max_stopped_wait_s": 1810.5,\n    "mean_switches": 1001.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "degraded",\n    "controller": "moving_only",\n    "runs": 4,\n    "mean_system_time_s": 963.29325,\n    "mean_completion": 0.9996265870052278,\n    "cleared_runs": 3,\n    "paired_reduction_vs_fixed_pct": 15.547850745419659,\n    "min_reduction_pct": 11.190540128781436,\n    "max_reduction_pct": 17.74247015192109,\n    "mean_max_stopped_wait_s": 1655.25,\n    "mean_switches": 962.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "degraded",\n    "controller": "predictive",\n    "runs": 4,\n    "mean_system_time_s": 659.563,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 42.11262316951276,\n    "min_reduction_pct": 39.15024007148601,\n    "max_reduction_pct": 44.68533450486238,\n    "mean_max_stopped_wait_s": 1434.5,\n    "mean_switches": 919.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "degraded",\n    "controller": "receiving_aware",\n    "runs": 4,\n    "mean_system_time_s": 1010.7339999999999,\n    "mean_completion": 0.9970047207833527,\n    "cleared_runs": 2,\n    "paired_reduction_vs_fixed_pct": 11.34140991511801,\n    "min_reduction_pct": 8.223026561318942,\n    "max_reduction_pct": 13.551511090439616,\n    "mean_max_stopped_wait_s": 1747.5,\n    "mean_switches": 1013.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "calibrated_pressure",\n    "runs": 4,\n    "mean_system_time_s": 1524.47875,\n    "mean_completion": 0.4985985454280747,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": -33.83992225696552,\n    "min_reduction_pct": -39.354324728678826,\n    "max_reduction_pct": -28.243699654564793,\n    "mean_max_stopped_wait_s": 2516.0,\n    "mean_switches": 1012.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "coordinated",\n    "runs": 4,\n    "mean_system_time_s": 980.80975,\n    "mean_completion": 0.9977628562378154,\n    "cleared_runs": 1,\n    "paired_reduction_vs_fixed_pct": 13.944096131452397,\n    "min_reduction_pct": 8.249973661686147,\n    "max_reduction_pct": 16.811436701336493,\n    "mean_max_stopped_wait_s": 1704.0,\n    "mean_switches": 1102.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "fixed",\n    "runs": 4,\n    "mean_system_time_s": 1140.28525,\n    "mean_completion": 0.8287020157334446,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": 0.0,\n    "min_reduction_pct": 0.0,\n    "max_reduction_pct": 0.0,\n    "mean_max_stopped_wait_s": 2178.5,\n    "mean_switches": 963,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "max_pressure",\n    "runs": 4,\n    "mean_system_time_s": 1735.642,\n    "mean_completion": 0.30343277678399455,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": -52.33344664038053,\n    "min_reduction_pct": -56.924865187943176,\n    "max_reduction_pct": -46.94744342455126,\n    "mean_max_stopped_wait_s": 2528.5,\n    "mean_switches": 910.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "message_only",\n    "runs": 4,\n    "mean_system_time_s": 1001.542,\n    "mean_completion": 0.9964437845127118,\n    "cleared_runs": 1,\n    "paired_reduction_vs_fixed_pct": 12.126598143915109,\n    "min_reduction_pct": 9.996383534013685,\n    "max_reduction_pct": 14.633694759400745,\n    "mean_max_stopped_wait_s": 1840.0,\n    "mean_switches": 1114.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "moving_only",\n    "runs": 4,\n    "mean_system_time_s": 984.71275,\n    "mean_completion": 0.9973823380501085,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": 13.578664880609697,\n    "min_reduction_pct": 10.243134433454562,\n    "max_reduction_pct": 17.59608075929603,\n    "mean_max_stopped_wait_s": 1862.0,\n    "mean_switches": 1095.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "predictive",\n    "runs": 4,\n    "mean_system_time_s": 696.303,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 38.88293080355259,\n    "min_reduction_pct": 35.68978326599743,\n    "max_reduction_pct": 42.36815050219536,\n    "mean_max_stopped_wait_s": 1437.0,\n    "mean_switches": 1029.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "receiving_aware",\n    "runs": 4,\n    "mean_system_time_s": 1022.512,\n    "mean_completion": 0.9931209662837456,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": 10.262811712735171,\n    "min_reduction_pct": 6.119831796384439,\n    "max_reduction_pct": 13.407392851257274,\n    "mean_max_stopped_wait_s": 1831.0,\n    "mean_switches": 1128,\n    "all_valid": true\n  },\n  {\n    "scenario": "directional_switch",\n    "load_scale": 1.0,\n    "regime": "perfect",\n    "controller": "sumo_actuated",\n    "runs": 4,\n    "mean_system_time_s": 1113.76325,\n    "mean_completion": 0.8302009518077405,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": 2.435715143636731,\n    "min_reduction_pct": -5.328971661595687,\n    "max_reduction_pct": 9.726648343083133,\n    "mean_max_stopped_wait_s": 2414.75,\n    "mean_switches": 839,\n    "all_valid": true\n  },\n  {\n    "scenario": "surge",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "calibrated_pressure",\n    "runs": 4,\n    "mean_system_time_s": 772.41975,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": -0.023761231160512764,\n    "min_reduction_pct": -11.052331792987344,\n    "max_reduction_pct": 12.422248066873479,\n    "mean_max_stopped_wait_s": 1294.75,\n    "mean_switches": 927,\n    "all_valid": true\n  },\n  {\n    "scenario": "surge",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "coordinated",\n    "runs": 4,\n    "mean_system_time_s": 564.40725,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 26.789877012196772,\n    "min_reduction_pct": 23.025024117271457,\n    "max_reduction_pct": 29.53834214951245,\n    "mean_max_stopped_wait_s": 1001.0,\n    "mean_switches": 767.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "surge",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "fixed",\n    "runs": 4,\n    "mean_system_time_s": 770.6569999999999,\n    "mean_completion": 0.9920134022355842,\n    "cleared_runs": 1,\n    "paired_reduction_vs_fixed_pct": 0.0,\n    "min_reduction_pct": 0.0,\n    "max_reduction_pct": 0.0,\n    "mean_max_stopped_wait_s": 1855.25,\n    "mean_switches": 956.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "surge",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "max_pressure",\n    "runs": 4,\n    "mean_system_time_s": 1315.4745,\n    "mean_completion": 0.8361146954865154,\n    "cleared_runs": 0,\n    "paired_reduction_vs_fixed_pct": -70.49805470068432,\n    "min_reduction_pct": -76.82037932972294,\n    "max_reduction_pct": -60.51501528340966,\n    "mean_max_stopped_wait_s": 2075.75,\n    "mean_switches": 1031.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "surge",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "message_only",\n    "runs": 4,\n    "mean_system_time_s": 587.446,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 23.79302294398562,\n    "min_reduction_pct": 21.965159344598888,\n    "max_reduction_pct": 27.192661015789575,\n    "mean_max_stopped_wait_s": 1014.75,\n    "mean_switches": 784,\n    "all_valid": true\n  },\n  {\n    "scenario": "surge",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "moving_only",\n    "runs": 4,\n    "mean_system_time_s": 574.33075,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 25.520770452607138,\n    "min_reduction_pct": 21.705836314534377,\n    "max_reduction_pct": 28.131663983490412,\n    "mean_max_stopped_wait_s": 1002.0,\n    "mean_switches": 765.5,\n    "all_valid": true\n  },\n  {\n    "scenario": "surge",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "predictive",\n    "runs": 4,\n    "mean_system_time_s": 430.78,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 44.147141401266374,\n    "min_reduction_pct": 40.983537658292654,\n    "max_reduction_pct": 45.61769630750755,\n    "mean_max_stopped_wait_s": 975.0,\n    "mean_switches": 793.75,\n    "all_valid": true\n  },\n  {\n    "scenario": "surge",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "receiving_aware",\n    "runs": 4,\n    "mean_system_time_s": 600.675,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 22.092902376908683,\n    "min_reduction_pct": 19.416418167026578,\n    "max_reduction_pct": 25.95107906160851,\n    "mean_max_stopped_wait_s": 1006.0,\n    "mean_switches": 785.25,\n    "all_valid": true\n  },\n  {\n    "scenario": "surge",\n    "load_scale": 0.6,\n    "regime": "perfect",\n    "controller": "sumo_actuated",\n    "runs": 4,\n    "mean_system_time_s": 687.304,\n    "mean_completion": 1.0,\n    "cleared_runs": 4,\n    "paired_reduction_vs_fixed_pct": 10.776742972420719,\n    "min_reduction_pct": 6.714970384266637,\n    "max_reduction_pct": 16.259633263812418,\n    "mean_max_stopped_wait_s": 1489.75,\n    "mean_switches": 834.5,\n    "all_valid": true\n  }\n]\n', 'reference/stage2_audit.json': '{\n  "evaluation_runs": 284,\n  "development_runs": 48,\n  "all_valid": true,\n  "matched_input_groups": 32,\n  "collision_events": 0,\n  "teleport_events": 0,\n  "all_metric_ledgers_agree": true,\n  "selected_before_evaluation": true\n}\n'}
for name,text in PAYLOAD.items():
    path=PROJECT/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(text,encoding="utf-8")
if shutil.which("g++") is None:
    raise RuntimeError("Use a Linux CPU runtime with g++, or install g++ locally.")
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(PROJECT/"requirements.txt")],check=True)
subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-v"],cwd=PROJECT,check=True)
print("Ready:",PROJECT)


In [ ]:
# Cell 2 — open the genuine live SUMO viewer. Press Start / reset run inside it.
PORT=int(os.environ.get("ADAPTIVE_SIGNALS_PORT","8765"))
if "LIVE_PROCESS" in globals() and LIVE_PROCESS.poll() is None:
    LIVE_PROCESS.terminate();LIVE_PROCESS.wait(timeout=10)
LIVE_LOG=(PROJECT/"live_server.log").open("w")
LIVE_PROCESS=subprocess.Popen([sys.executable,"-u",str(PROJECT/"live_server.py"),"--port",str(PORT)],
                              cwd=PROJECT,stdout=LIVE_LOG,stderr=subprocess.STDOUT)
ready=False
for _ in range(100):
    if LIVE_PROCESS.poll() is not None:break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/",timeout=1) as response:
            ready=b"Adaptive Signals" in response.read()
        if ready:break
    except OSError:time.sleep(.1)
if not ready:
    raise RuntimeError("The live server did not start. "+(PROJECT/"live_server.log").read_text()[-3000:])
print("Live simulator ready. Click Start / reset run. Keep this runtime connected.")
try:
    from google.colab import output
except ImportError:
    print(f"Open http://127.0.0.1:{PORT}/ in a local browser.")
else:
    output.serve_kernel_port_as_iframe(PORT,height="1080",cache_in_notebook=False)


In [ ]:
# Cell 3 — optional benchmark reproduction and export.
# Leave False for immediate live use. Set True to rerun the complete experiment.
RUN_BENCHMARK=os.environ.get("ADAPTIVE_SIGNALS_RUN_BENCHMARK","0")=="1"
PROFILE=os.environ.get("ADAPTIVE_SIGNALS_PROFILE","full")  # full or smoke
EXPORT_RESULTS=RUN_BENCHMARK
if RUN_BENCHMARK:
    OUTPUT=PROJECT/"runs"/("reproduced_"+PROFILE)
    subprocess.run([sys.executable,"-u",str(PROJECT/"stage2.py"),"--profile",PROFILE,
                    "--output",str(OUTPUT),"--resume"],cwd=PROJECT,check=True)
    rows=json.loads((OUTPUT/"aggregate.json").read_text())
else:
    rows=json.loads((PROJECT/"reference/stage2_aggregate.json").read_text())
    print("Showing the saved accepted Stage-2 summary; no new benchmark was run.")
lines=["| Demand | Load | Sensors | Discharge system time (s/request) | Reduction vs fixed | Cleared |",
       "|---|---:|---|---:|---:|---:|"]
for r in rows:
    if r["controller"]=="predictive":
        lines.append(f"| {r['scenario']} | {r['load_scale']:g} | {r['regime']} | {r['mean_system_time_s']:.1f} | {r['paired_reduction_vs_fixed_pct']:.1f}% | {r['cleared_runs']}/{r['runs']} |")
try:
    from IPython.display import display, Markdown
except ImportError:
    print("\n".join(lines))
else:
    display(Markdown("\n".join(lines)))
seed_note="Smoke profile: functionality check only." if RUN_BENCHMARK and PROFILE=="smoke" else "Four fresh evaluation seeds."
print("Synthetic network.",seed_note,"No RL, live CCTV, or real-city efficacy claim.")
if EXPORT_RESULTS:
    archive=BASE/("AdaptiveSignals_Stage2_"+PROFILE+"_Results.zip")
    with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
        for path in PROJECT.rglob("*"):
            if not path.is_file():continue
            rel=path.relative_to(PROJECT)
            if "__pycache__" in rel.parts or "build" in rel.parts or str(rel).startswith("runs/live/"):continue
            if path.suffix in [".tmp",".log"]:continue
            z.write(path,Path(PROJECT.name)/rel)
    print("Source and completed benchmark results:",archive)
    try:
        from google.colab import files
    except ImportError:pass
    else:files.download(str(archive))
